# What This Code Actually Does: End-to-End Pipeline

## The Big Picture

You're building a system that:
1. **Detects lab objects** (pipettes, tubes, racks, etc.) in FineBio videos using YOLO-26
2. **Fuses detections across multiple camera views** (FPV + TPV) to handle occlusions
3. **Predicts which step** of a biological experiment is happening at each moment

---

## Step-by-Step Breakdown

### 1. **FineBio Dataset Setup** (Cells 4-7)
- Loads FineBio annotations: 35 object classes (pipettes, tubes, hands, etc.)
- Parses action files (`P*.txt`) that tell you: "At time 24.96-39.85s, the step is `add_pbs`, the verb is `insert`, the manipulated object is `blue_pipette`"

### 2. **Train YOLO-26 on FineBio Objects** (Cells 13-16)
- Converts FineBio COCO annotations → YOLO format
- Fine-tunes YOLO-26 to recognize **35 FineBio lab objects** instead of generic COCO classes
- Saves trained weights: `runs/.../yolo26n_joint3/weights/best.pt`

### 3. **Multi-View Detection** (Cell 19)
For each experiment instance (e.g., `P03_02_01`):
- Finds **FPV video** (first-person view) and **TPV video** (third-person view)
- At each timestamp (1 fps):
  - Runs YOLO-26 on **both FPV and TPV frames**
  - Gets object counts from each view: `{'blue_pipette': 2, '50ml_tube': 1, ...}`
  - **Fuses counts**: `counts_fused = counts_fpv + counts_tpv` (handles occlusion)
  - Aligns with FineBio action annotation: "At this time, step=`add_pbs`"

### 4. **Feature Extraction** (Cells 20, 28)
Turns object counts into **feature vectors**:
- **Original approach**: Just raw counts → `[2, 0, 1, 0, ...]` (35 dims)
- **Rich features**: Counts + temporal deltas + hand presence + diversity → `[2, 0, 1, ..., +0.5, -1, ..., 0.8, 0.3, 0.15]` (73 dims)

### 5. **Step Recognition Model** (Cells 22-23, 29)
- **Input**: Sequence of feature vectors over time `(T, 73)`
- **Model**: Bidirectional GRU (2 layers, hidden=256) → Linear layer
- **Output**: Per-frame step predictions (23 possible step classes)
- **Training**: Cross-entropy loss with class weights (handles imbalanced steps)

---

## What's Actually Happening During Inference

1. **Video frame** → YOLO-26 detects objects → `{'blue_pipette': 1, 'left_hand': 1, '50ml_tube': 2}`
2. **Counts** → Feature vector → `[0, 1, 0, ..., 1, 0, 2, ...]` (73 dims with deltas/hands)
3. **Feature sequence** → GRU processes temporal context → Hidden states
4. **Hidden states** → Linear layer → Logits for 23 step classes
5. **Argmax** → Predicted step: `"add_pbs"`

---

## Why Accuracy Might Be Low

1. **Raw counts are weak**: Just knowing "2 pipettes, 3 tubes" doesn't tell you **what action** is happening
2. **Temporal alignment**: Action annotations might not perfectly align with video frames
3. **Class imbalance**: Some steps are rare (e.g., `dispense_solution` = 1.1% of frames)
4. **Missing context**: You're not using:
   - **Spatial relationships** (which objects are near each other)
   - **Actual manipulation detection** (which object is in hand, not just "hand + object both present")
   - **Longer temporal windows** (maybe need 5-10 seconds of context, not just 1 frame)

---

## What You're Missing for Better Accuracy

1. **Manipulated object detection**: Use hand-object proximity from YOLO bboxes to identify which object is actually being manipulated
2. **Spatial features**: Distance/overlap between hand bbox and object bboxes
3. **Longer temporal context**: Use windows of 5-10 frames, not just per-frame
4. **Atomic operations as intermediate signals**: Train on `(verb, manipulated_object)` pairs first, then use those to predict steps

# Imports

In [172]:
import ultralytics
import cv2
import numpy as np 
import matplotlib.pyplot as plt 
import pickle 
import pandas as pd  
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torchvision 
import torchvision.transforms as transforms 
import torchvision.datasets as datasets 
import torchvision.models as models 

DATASET_PATH="/home/nyan/FineBio"


In [173]:
model=ultralytics.YOLO("yolo26n.pt")


# FineBio + YOLO-26: multi-view detection and atomic/step recognition

In this section we:
- align with the FineBio dataset on disk,
- run YOLO-26 for object / manipulated-object detection,
- build symbolic features over time,
- and set up a simple temporal model for atomic operations and steps.


In [174]:
from pathlib import Path
import json
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

DATASET_PATH = Path("/home/nyan/FineBio")
ANNOTATIONS_DIR = DATASET_PATH / "01 annotations"
COCO_ZIP = ANNOTATIONS_DIR / "finebio_coco_annotations.zip"
ACTION_ZIP = ANNOTATIONS_DIR / "finebio_action_annotations.zip"

IMAGES_ROOT = DATASET_PATH / "12 finebio_object_detection_images" / "finebio_object_detection_images"

print("Dataset root:", DATASET_PATH)
print("Images root:", IMAGES_ROOT)
print("Annotations dir:", ANNOTATIONS_DIR)


Dataset root: /home/nyan/FineBio
Images root: /home/nyan/FineBio/12 finebio_object_detection_images/finebio_object_detection_images
Annotations dir: /home/nyan/FineBio/01 annotations


In [175]:
import zipfile

# Helper to ensure COCO annotations are extracted next to the zip
COCO_DIR = ANNOTATIONS_DIR / "finebio_coco_annotations"
ACTION_DIR = ANNOTATIONS_DIR / "finebio_action_annotations"

if not COCO_DIR.exists() and COCO_ZIP.exists():
    with zipfile.ZipFile(COCO_ZIP, "r") as zf:
        zf.extractall(ANNOTATIONS_DIR)
        print("Extracted COCO annotations to", COCO_DIR)

if not ACTION_DIR.exists() and ACTION_ZIP.exists():
    with zipfile.ZipFile(ACTION_ZIP, "r") as zf:
        zf.extractall(ANNOTATIONS_DIR)
        print("Extracted action annotations to", ACTION_DIR)

print("COCO_DIR exists:", COCO_DIR.exists())
print("ACTION_DIR exists:", ACTION_DIR.exists())


COCO_DIR exists: True
ACTION_DIR exists: True


In [176]:
# Load one of the FineBio COCO-style annotation files (joint train+val as default)
COCO_JSON = COCO_DIR / "v1_trainval_joint.json"

with open(COCO_JSON, "r") as f:
    coco_data = json.load(f)

categories = {c["id"]: c["name"] for c in coco_data["categories"]}
cat_name_to_id = {v: k for k, v in categories.items()}

print("Loaded COCO annotations from", COCO_JSON)
print("Number of categories:", len(categories))
print("Sample categories:", list(categories.values())[:10])

# Build quick image-id -> file-name mapping
image_id_to_info = {img["id"]: img for img in coco_data["images"]}
file_name_to_image_id = {img["file_name"]: img["id"] for img in coco_data["images"]}

print("Number of images in COCO JSON:", len(image_id_to_info))


Loaded COCO annotations from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_trainval_joint.json
Number of categories: 35
Sample categories: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Number of images in COCO JSON: 1632


In [177]:
# Parse action (atomic operation + step) annotations for a given experiment instance

@dataclass
class FineBioAction:
    start_sec: float
    end_sec: float
    task: str  # coarse step-level task (e.g., remove_culture_medium)
    hand_side: Optional[str]
    verb: Optional[str]
    manipulated_object: Optional[str]
    affected_object: Optional[str]


def load_actions_for_instance(instance_id: str) -> List[FineBioAction]:
    """Load atomic operations + step-like tasks for one experiment instance.

    Args:
        instance_id: e.g. "P03_02_01" corresponding to FineBio naming.
    """
    txt_path = ACTION_DIR / f"{instance_id}.txt"
    actions: List[FineBioAction] = []
    if not txt_path.exists():
        print("No action annotation file:", txt_path)
        return actions

    with open(txt_path, "r") as f:
        header = f.readline().strip().split(",")
        # Expect: start_sec,end_sec,task,hand_side,verb,manipulated_object,affected_object
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 2:
                continue
            # Pad to length 7
            while len(parts) < 7:
                parts.append("")
            start_sec, end_sec = float(parts[0]), float(parts[1])
            task, hand_side, verb, manipulated_object, affected_object = parts[2:7]
            actions.append(
                FineBioAction(
                    start_sec=start_sec,
                    end_sec=end_sec,
                    task=task or "",
                    hand_side=hand_side or None,
                    verb=verb or None,
                    manipulated_object=manipulated_object or None,
                    affected_object=affected_object or None,
                )
            )
    print(f"Loaded {len(actions)} actions for {instance_id}")
    return actions


# Quick sanity check on one instance from the paper example
_example_instance = "P03_02_01"
_example_actions = load_actions_for_instance(_example_instance)
_example_actions[:5]


Loaded 169 actions for P03_02_01


[FineBioAction(start_sec=1.0, end_sec=24.92, task='remove_culture_medium', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=24.96, end_sec=39.85, task='add_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=39.87, end_sec=44.9, task='shake_plate', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=44.97, end_sec=56.07, task='aspirate_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None),
 FineBioAction(start_sec=56.16, end_sec=69.16, task='add_pbs', hand_side=None, verb=None, manipulated_object=None, affected_object=None)]

In [178]:
# YOLO-26 based object detection and symbolic feature extraction

from collections import Counter

# model is already created above as `model = ultralytics.YOLO("yolo26n.pt")`
# We assume it has been (or will be) fine-tuned on FineBio categories so that
# `model.names` is aligned with object classes like blue_pipette, blue_tip, etc.

model_names: Dict[int, str] = model.names  # id -> name


def yolo_features_for_image(img_path: Path,
                            conf: float = 0.25) -> Tuple[Counter, List[Dict]]:
    """Run YOLO-26 on one image and return
    - a Counter over predicted class names (symbolic feature),
    - the raw detection records (for debugging / visualization).
    """
    results = model(str(img_path), conf=conf, verbose=False)
    if not results:
        return Counter(), []
    res = results[0]
    detections: List[Dict] = []
    counts: Counter = Counter()

    if res.boxes is None:
        return counts, detections

    for b in res.boxes:
        cls_id = int(b.cls.item())
        score = float(b.conf.item())
        name = model_names.get(cls_id, str(cls_id))
        counts[name] += 1
        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
        detections.append({
            "class_id": cls_id,
            "class_name": name,
            "score": score,
            "bbox": [x1, y1, x2, y2],
        })

    return counts, detections


# Example on one FineBio object-detection image
from itertools import islice

sample_images = list(islice(IMAGES_ROOT.glob("*.jpg"), 5))
print("Found", len(sample_images), "sample images")
if sample_images:
    c, dets = yolo_features_for_image(sample_images[0])
    print("Sample image:", sample_images[0].name)
    print("Symbolic feature (counts per class):", c)
    print("Num detections:", len(dets))


Found 5 sample images
Sample image: P28_05_01_000407.jpg
Symbolic feature (counts per class): Counter({'keyboard': 1, 'person': 1, 'book': 1})
Num detections: 3


In [179]:
# From per-frame YOLO features to a temporal sequence for atomic operations / steps

import math


def assign_action_label(time_sec: float, actions: List[FineBioAction]) -> Optional[FineBioAction]:
    """Find which action interval (if any) covers this timestamp."""
    for a in actions:
        if a.start_sec <= time_sec <= a.end_sec:
            return a
    return None


def resolve_video_paths(instance_id: str, view: str = "tpv") -> List[Path]:
    """Return all matching video paths for a given instance id and view.

    - TPV videos live in 05/06/08 .../finebio_videos and use Pxx_xx_xx_T1..T5.mp4
    - FPV videos live in 09/11 .../finebio_videos and 04 (test),
      and can be either Pxx_xx_xx.mp4 or Pxx_xx_xx_T1..T5.mp4.
    """
    paths: List[Path] = []

    if view == "tpv":
        tpv_dirs = [
            DATASET_PATH / "05 finebio_videos_tpv_train" / "finebio_videos",
            DATASET_PATH / "06 finebio_videos_tpv_valid" / "finebio_videos",
            DATASET_PATH / "08 finebio_videos_tpv_test" / "finebio_videos",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in tpv_dirs:
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    elif view == "fpv":
        fpv_dirs = [
            DATASET_PATH / "09 finebio_videos_fpv_train" / "finebio_videos",
            DATASET_PATH / "11 finebio_videos_fpv_valid" / "finebio_videos",
            DATASET_PATH / "04 finebio_videos_fpv_test",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in fpv_dirs:
            # single-clip version
            cand_single = d / f"{instance_id}.mp4"
            if cand_single.exists():
                paths.append(cand_single)
            # trial versions
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    else:
        raise ValueError(f"Unknown view: {view}")

    return paths


def build_sequence_for_instance(instance_id: str,
                                fps: float = 1.0,
                                view: str = "tpv",
                                max_frames: Optional[int] = None,
                                conf: float = 0.25) -> List[Dict]:
    """Construct a time sequence of symbolic YOLO features + labels for one instance.

    Returns a list of dicts, each containing:
        - time_sec
        - yolo_class_counts (dict[class_name -> count])
        - detections (raw YOLO detections)
        - action (FineBioAction or None)
    """
    from collections import Counter

    actions = load_actions_for_instance(instance_id)
    if not actions:
        return []

    video_paths = resolve_video_paths(instance_id, view=view)
    if not video_paths:
        print(f"No video found for {instance_id} (view={view})")
        return []

    # For now, just use the first matching video
    video_path = video_paths[0]
    print("Using video:", video_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print("Failed to open video", video_path)
        return []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    duration_sec = total_frames / video_fps if video_fps > 0 else 0.0
    print(f"Video FPS={video_fps:.2f}, frames={total_frames}, duration={duration_sec:.2f}s")

    frame_interval = max(int(round(video_fps / fps)), 1)
    seq: List[Dict] = []

    frame_idx = 0
    sampled = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            time_sec = frame_idx / video_fps

            # Run YOLO directly on the numpy frame (OpenCV BGR is fine)
            results = model(frame, conf=conf, verbose=False)
            counts = Counter()
            dets: List[Dict] = []
            if results:
                res = results[0]
                if res.boxes is not None:
                    for b in res.boxes:
                        cls_id = int(b.cls.item())
                        score = float(b.conf.item())
                        name = model_names.get(cls_id, str(cls_id))
                        counts[name] += 1
                        x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
                        dets.append({
                            "class_id": cls_id,
                            "class_name": name,
                            "score": score,
                            "bbox": [x1, y1, x2, y2],
                        })

            action = assign_action_label(time_sec, actions)
            seq.append({
                "time_sec": time_sec,
                "yolo_class_counts": dict(counts),
                "detections": dets,
                "action": action,
            })

            sampled += 1
            if max_frames is not None and sampled >= max_frames:
                break

        frame_idx += 1

    cap.release()
    print(f"Built sequence of length {len(seq)} for {instance_id} ({view})")
    return seq


# Skeleton for a small temporal model (you can refine/extend this later)

class SimpleGRUHead(nn.Module):
    """A lightweight GRU-based classifier over YOLO-derived symbolic features.

    You would feed sequences of per-frame feature vectors (e.g. class-counts
    embedded into a fixed-size vector) and train it to predict atomic
    operations or steps.
    """

    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int, num_layers: int = 1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (B, T, D)
        h, _ = self.gru(x)
        logits = self.fc(h)  # (B, T, num_classes)
        return logits


### What you can run when you’re tired

- Run all cells up to the bottom: this wires FineBio, loads annotations, and creates the YOLO-26 model.
- Call `build_sequence_for_instance("P03_02_01", fps=1.0, view="fpv", max_frames=30)` to get a ready-made sequence with YOLO counts + atomic/step labels.
- You can then plug that into any temporal model (like `SimpleGRUHead`) when you feel like training; the heavy wiring work is already done.


# Use Ultralytics Yolo 26


# Train YOLO-26 on FineBio object detection (COCO annotations)

We first fine-tune YOLO-26 on the FineBio COCO annotations so it learns lab objects
(blue_pipette, tips, tubes, racks, etc.) instead of generic COCO classes (person, laptop, bottle).

In [180]:
# Prepare YOLO-format dataset from FineBio COCO JSONs (train/val)

import os
from collections import defaultdict

FINEBIO_YOLO_ROOT = Path("/home/nyan/k-project/finebio_yolo_joint")
FINEBIO_YOLO_ROOT.mkdir(parents=True, exist_ok=True)

IMAGES_SRC = IMAGES_ROOT  # original FineBio object detection images

TRAIN_JSON = COCO_DIR / "v1_train_joint.json"
VAL_JSON = COCO_DIR / "v1_valid_joint.json"

images_train_dir = FINEBIO_YOLO_ROOT / "images" / "train"
images_val_dir = FINEBIO_YOLO_ROOT / "images" / "val"
labels_train_dir = FINEBIO_YOLO_ROOT / "labels" / "train"
labels_val_dir = FINEBIO_YOLO_ROOT / "labels" / "val"

for d in [images_train_dir, images_val_dir, labels_train_dir, labels_val_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Load categories from train JSON and define class mapping (COCO id -> 0-based index)
with open(TRAIN_JSON, "r") as f:
    train_coco = json.load(f)

categories_joint = sorted(train_coco["categories"], key=lambda c: c["id"])
cat_id_to_idx = {c["id"]: i for i, c in enumerate(categories_joint)}
finebio_class_names = [c["name"] for c in categories_joint]
print("Number of FineBio classes:", len(finebio_class_names))
print("First few classes:", finebio_class_names[:10])


def prepare_split(coco_json_path: Path, images_dest: Path, labels_dest: Path):
    print("Preparing split from", coco_json_path)
    with open(coco_json_path, "r") as f:
        data = json.load(f)

    images = {img["id"]: img for img in data["images"]}
    anns_by_image: Dict[int, list] = defaultdict(list)
    for ann in data["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    num_images = len(images)
    num_anns = len(data["annotations"])
    print(f"  images={num_images}, annotations={num_anns}")

    for img_id, img in images.items():
        fname = img["file_name"]
        width, height = img["width"], img["height"]
        src_img = IMAGES_SRC / fname
        if not src_img.exists():
            # Some images may be missing; skip gracefully
            continue

        dst_img = images_dest / fname
        if not dst_img.exists():
            try:
                os.symlink(src_img, dst_img)
            except OSError:
                # Fallback to copy if symlink is not permitted
                import shutil
                shutil.copy2(src_img, dst_img)

        anns = anns_by_image.get(img_id, [])
        if not anns:
            # No annotations -> background image; YOLO is fine with missing txt
            continue

        label_path = labels_dest / (Path(fname).stem + ".txt")
        with open(label_path, "w") as lf:
            for ann in anns:
                cat_id = ann["category_id"]
                if cat_id not in cat_id_to_idx:
                    continue
                cls_idx = cat_id_to_idx[cat_id]
                x, y, w, h = ann["bbox"]  # COCO: top-left x,y,width,height
                cx = (x + w / 2.0) / width
                cy = (y + h / 2.0) / height
                nw = w / width
                nh = h / height
                lf.write(f"{cls_idx} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")


prepare_split(TRAIN_JSON, images_train_dir, labels_train_dir)
prepare_split(VAL_JSON, images_val_dir, labels_val_dir)

print("Finished preparing YOLO-format dataset at", FINEBIO_YOLO_ROOT)

Number of FineBio classes: 35
First few classes: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Preparing split from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_train_joint.json
  images=1394, annotations=50886
Preparing split from /home/nyan/FineBio/01 annotations/finebio_coco_annotations/v1_valid_joint.json
  images=238, annotations=8867
Finished preparing YOLO-format dataset at /home/nyan/k-project/finebio_yolo_joint


In [181]:
# Create a Ultralytics data.yaml for FineBio joint object detection

data_yaml_path = FINEBIO_YOLO_ROOT / "finebio_joint.yaml"

names_lines = "\n".join([f"  {i}: {name}" for i, name in enumerate(finebio_class_names)])

yaml_text = f"""path: {FINEBIO_YOLO_ROOT}
train: images/train
val: images/val
names:
{names_lines}
"""

with open(data_yaml_path, "w") as f:
    f.write(yaml_text)

print("Wrote data.yaml to", data_yaml_path)
print("\nPreview:\n", yaml_text)

Wrote data.yaml to /home/nyan/k-project/finebio_yolo_joint/finebio_joint.yaml

Preview:
 path: /home/nyan/k-project/finebio_yolo_joint
train: images/train
val: images/val
names:
  0: left_hand
  1: right_hand
  2: blue_pipette
  3: yellow_pipette
  4: red_pipette
  5: 8_channel_pipette
  6: blue_tip
  7: yellow_tip
  8: red_tip
  9: 8_channel_tip
  10: blue_tip_rack
  11: yellow_tip_rack
  12: red_tip_rack
  13: 8_channel_tip_rack
  14: 50ml_tube
  15: 15ml_tube
  16: micro_tube
  17: 8_tube_stripes
  18: 8_tube_stripes_lid
  19: 50ml_tube_rack
  20: 15ml_tube_rack
  21: micro_tube_rack
  22: 8_tube_stripes_rack
  23: 8_tube_stripes_rack_lid
  24: cell_culture_plate
  25: cell_culture_plate_lid
  26: trash_can
  27: centrifuge
  28: vortex_mixer
  29: magnetic_rack
  30: pcr_machine
  31: tube_with_spin_column
  32: spin_column
  33: tube_without_lid
  34: pen



In [182]:
# # Fine-tune YOLO-26 on FineBio joint object detection

# finebio_data_yaml = str(data_yaml_path)

# # Start from the generic YOLO-26 weights you already loaded
# # (model variable defined above as ultralytics.YOLO("yolo26n.pt"))

# print("Training YOLO-26 on FineBio... this may take a while.")

# train_results = model.train(
#     data=finebio_data_yaml,
#     epochs=30,
#     imgsz=640,
#     batch=16,
#     project="runs/finebio_yolo26",
#     name="yolo26n_joint",
#     patience=5,
#     device=0,  # use GPU 0 explicitly
# )

# print("Training finished.")

# from pathlib import Path
# save_dir = Path(model.trainer.save_dir)  # e.g. runs/finebio_yolo26/yolo26n_joint
# best_weights = save_dir / "weights" / "best.pt"
# print("Best weights saved at:", best_weights)

In [183]:
# Reload YOLO-26 with FineBio-trained weights and sanity-check detections

from pathlib import Path

# Hard-code the latest FineBio-trained checkpoint so you never have to retrain just to get the path
best_weights = Path('/home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt')
print('Using FineBio weights:', best_weights)

# Reload model from the best FineBio weights
model = ultralytics.YOLO(str(best_weights))
model_names = model.names
print("Loaded FineBio-trained YOLO-26. Number of classes:", len(model_names))
print("First few classes:", list(model_names.values())[:10])

# Quick check on one FineBio object-detection image
from itertools import islice

sample_images = list(islice(IMAGES_ROOT.glob("*.jpg"), 5))
print("Found", len(sample_images), "sample images for sanity check")
if sample_images:
    c, dets = yolo_features_for_image(sample_images[0], conf=0.25)
    print("Sample image:", sample_images[0].name)
    print("Symbolic feature (counts per class):", c)
    print("Num detections:", len(dets))


Using FineBio weights: /home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt
Loaded FineBio-trained YOLO-26. Number of classes: 35
First few classes: ['left_hand', 'right_hand', 'blue_pipette', 'yellow_pipette', 'red_pipette', '8_channel_pipette', 'blue_tip', 'yellow_tip', 'red_tip', '8_channel_tip']
Found 5 sample images for sanity check
Sample image: P28_05_01_000407.jpg
Symbolic feature (counts per class): Counter({'8_channel_tip_rack': 4, 'vortex_mixer': 2, '50ml_tube_rack': 2, 'centrifuge': 2, 'micro_tube': 2, '8_tube_stripes': 2, 'trash_can': 1, 'magnetic_rack': 1, 'cell_culture_plate': 1, 'pcr_machine': 1, 'right_hand': 1, '8_tube_stripes_rack_lid': 1, 'left_hand': 1, 'red_tip_rack': 1, '8_tube_stripes_rack': 1, '8_channel_pipette': 1, 'micro_tube_rack': 1, 'yellow_tip_rack': 1})
Num detections: 26


In [184]:
# Quick demo: build a short FPV sequence and inspect first few entries

example_seq = build_sequence_for_instance("P03_02_01", fps=1.0, view="fpv", max_frames=10, conf=0.25)
print("Sequence length:", len(example_seq))
for step in example_seq[:5]:
    a = step["action"]
    print(f"t={step['time_sec']:.2f}s, counts={step['yolo_class_counts']}, "
          f"task={a.task if a else None}, verb={a.verb if a else None}, obj={a.manipulated_object if a else None}")


Loaded 169 actions for P03_02_01
Using video: /home/nyan/FineBio/04 finebio_videos_fpv_test/P03_02_01.mp4
Video FPS=29.97, frames=4613, duration=153.92s
Built sequence of length 10 for P03_02_01 (fpv)
Sequence length: 10
t=0.00s, counts={'centrifuge': 1, 'blue_tip_rack': 1, 'yellow_tip_rack': 1, 'cell_culture_plate': 1, 'micro_tube': 5, 'micro_tube_rack': 1, '50ml_tube_rack': 2, '8_channel_tip_rack': 1, '50ml_tube': 1, 'trash_can': 1, 'red_pipette': 1}, task=None, verb=None, obj=None
t=1.00s, counts={'magnetic_rack': 1, 'blue_tip_rack': 1, 'centrifuge': 1, 'trash_can': 1, '8_channel_tip_rack': 2, 'red_tip_rack': 1, 'cell_culture_plate': 1, '8_channel_pipette': 2, 'left_hand': 2, 'micro_tube': 8, 'pcr_machine': 2, '50ml_tube_rack': 3, 'yellow_tip_rack': 2, '8_tube_stripes_rack': 3, 'right_hand': 1}, task=remove_culture_medium, verb=None, obj=None
t=2.00s, counts={'vortex_mixer': 1, 'cell_culture_plate': 1, 'magnetic_rack': 1, 'centrifuge': 1, '8_tube_stripes_rack': 1, '50ml_tube_rack': 

# Multi-view YOLO-26 + atomic operation / step sequences

Now that YOLO-26 is fine-tuned on FineBio, we build **multi-view** sequences (FPV + TPV)
with fused object detections aligned to atomic operations and steps.

In [185]:
# Helper: run YOLO-26 on a raw frame (numpy array) and return class-counts + detections

from collections import Counter


def yolo_counts_from_frame(frame, conf: float = 0.25):
    """Run YOLO on a single BGR frame and return (counts, detections)."""
    results = model(frame, conf=conf, verbose=False)
    counts = Counter()
    dets: List[Dict] = []
    if results:
        res = results[0]
        if res.boxes is not None:
            for b in res.boxes:
                cls_id = int(b.cls.item())
                score = float(b.conf.item())
                name = model_names.get(cls_id, str(cls_id))
                counts[name] += 1
                x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
                dets.append({
                    "class_id": cls_id,
                    "class_name": name,
                    "score": score,
                    "bbox": [x1, y1, x2, y2],
                })
    return counts, dets


def build_multiview_sequence_for_instance(instance_id: str,
                                          fps: float = 1.0,
                                          max_frames: Optional[int] = None,
                                          conf: float = 0.25) -> List[Dict]:
    """FPV + TPV fused YOLO features + labels for one experiment instance.

    For each sampled timestamp, we:
    - grab the nearest FPV frame
    - grab the nearest TPV frame (first available TPV camera)
    - run YOLO on both
    - fuse counts across views
    - attach the aligned FineBioAction (atomic/step).
    """
    actions = load_actions_for_instance(instance_id)
    if not actions:
        return []

    fpv_paths = resolve_video_paths(instance_id, view="fpv")
    tpv_paths = resolve_video_paths(instance_id, view="tpv")
    if not fpv_paths or not tpv_paths:
        print("Missing FPV or TPV video for", instance_id)
        return []

    fpv_path = fpv_paths[0]
    tpv_path = tpv_paths[0]
    print("Using FPV:", fpv_path)
    print("Using TPV:", tpv_path)

    cap_fpv = cv2.VideoCapture(str(fpv_path))
    cap_tpv = cv2.VideoCapture(str(tpv_path))
    if not cap_fpv.isOpened() or not cap_tpv.isOpened():
        print("Failed to open FPV or TPV video")
        return []

    fps_fpv = cap_fpv.get(cv2.CAP_PROP_FPS) or 30.0
    fps_tpv = cap_tpv.get(cv2.CAP_PROP_FPS) or 30.0
    n_fpv = int(cap_fpv.get(cv2.CAP_PROP_FRAME_COUNT))
    n_tpv = int(cap_tpv.get(cv2.CAP_PROP_FRAME_COUNT))
    dur_fpv = n_fpv / fps_fpv if fps_fpv > 0 else 0.0
    dur_tpv = n_tpv / fps_tpv if fps_tpv > 0 else 0.0
    duration = min(dur_fpv, dur_tpv)
    print(f"FPV: fps={fps_fpv:.2f}, frames={n_fpv}, dur={dur_fpv:.2f}s")
    print(f"TPV: fps={fps_tpv:.2f}, frames={n_tpv}, dur={dur_tpv:.2f}s")

    seq: List[Dict] = []
    step = 1.0 / fps
    t = 0.0
    sampled = 0
    while t <= duration:
        # Seek FPV and TPV to time t (ms)
        cap_fpv.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        cap_tpv.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        ok_fpv, frame_fpv = cap_fpv.read()
        ok_tpv, frame_tpv = cap_tpv.read()
        if not ok_fpv or not ok_tpv:
            break

        counts_fpv, dets_fpv = yolo_counts_from_frame(frame_fpv, conf=conf)
        counts_tpv, dets_tpv = yolo_counts_from_frame(frame_tpv, conf=conf)

        fused_counts = Counter(counts_fpv)
        for k, v in counts_tpv.items():
            fused_counts[k] += v

        action = assign_action_label(t, actions)
        seq.append({
            "time_sec": t,
            "counts_fpv": dict(counts_fpv),
            "counts_tpv": dict(counts_tpv),
            "counts_fused": dict(fused_counts),
            "dets_fpv": dets_fpv,
            "dets_tpv": dets_tpv,
            "action": action,
        })

        sampled += 1
        if max_frames is not None and sampled >= max_frames:
            break
        t += step

    cap_fpv.release()
    cap_tpv.release()
    print(f"Built multiview sequence of length {len(seq)} for {instance_id}")
    return seq

In [186]:
# Turn fused YOLO counts into feature vectors + step/atomic labels

import numpy as np

# Use FineBio class names from the COCO categories / YOLO training
finebio_class_list = finebio_class_names  # length 35
name_to_index = {name: i for i, name in enumerate(finebio_class_list)}
feature_dim = len(finebio_class_list)
print("Feature dim (num FineBio classes):", feature_dim)


def counts_to_vector(counts: Dict[str, int]) -> np.ndarray:
    v = np.zeros(feature_dim, dtype=np.float32)
    for name, c in counts.items():
        idx = name_to_index.get(name)
        if idx is not None:
            v[idx] = float(c)
    return v


def multiview_sequence_to_arrays(seq: List[Dict]) -> Tuple[np.ndarray, np.ndarray]:
    """Use fused counts to build (X, y_task) for one sequence.

    - X: (T, D) fused YOLO counts per time step
    - y: (T,) integer step labels from `action.task` (-1 if no label)
    """
    T = len(seq)
    X = np.zeros((T, feature_dim), dtype=np.float32)
    y = np.full((T,), -1, dtype=np.int64)

    task_to_id: Dict[str, int] = {}
    next_id = 0

    for t, step in enumerate(seq):
        X[t] = counts_to_vector(step["counts_fused"])
        a = step["action"]
        if a and a.task:
            if a.task not in task_to_id:
                task_to_id[a.task] = next_id
                next_id += 1
            y[t] = task_to_id[a.task]

    print("Discovered", len(task_to_id), "distinct step tasks in this sequence")
    return X, y


# Example: build multiview sequence + arrays for one instance (no training yet)

mv_seq = build_multiview_sequence_for_instance("P03_02_01", fps=1.0, max_frames=20, conf=0.25)
print("Multiview sequence length:", len(mv_seq))
if mv_seq:
    X_mv, y_mv = multiview_sequence_to_arrays(mv_seq)
    print("X_mv shape:", X_mv.shape, "y_mv shape:", y_mv.shape)
    print("First 5 labels:", y_mv[:5])

Feature dim (num FineBio classes): 35
Loaded 169 actions for P03_02_01
Using FPV: /home/nyan/FineBio/04 finebio_videos_fpv_test/P03_02_01.mp4
Using TPV: /home/nyan/FineBio/08 finebio_videos_tpv_test/finebio_videos/P03_02_01_T1.mp4
FPV: fps=29.97, frames=4613, dur=153.92s
TPV: fps=29.97, frames=4613, dur=153.92s
Built multiview sequence of length 20 for P03_02_01
Multiview sequence length: 20
Discovered 1 distinct step tasks in this sequence
X_mv shape: (20, 35) y_mv shape: (20,)
First 5 labels: [-1  0  0  0  0]


## Build a multi-instance dataset for step recognition

Here we aggregate **multiview YOLO-26 sequences for all FineBio instances** and
build a dataset `(X, y)` suitable for training a temporal model to predict **steps**.

In [187]:
# # Collect multiview sequences and build global step labels

# from tqdm.auto import tqdm


# def build_multiview_dataset(fps: float = 1.0,
#                             max_frames: int = 60,
#                             conf: float = 0.25):
#     """Build a dataset of multiview sequences across all annotated instances.

#     Returns:
#         X_list: list of (T_i, D) numpy arrays of fused features
#         y_list: list of (T_i,) int arrays of step labels (-1 for unlabeled frames)
#         seq_instance_ids: list of instance_id strings
#         task_to_id: dict mapping step task string -> label index
#     """
#     # 1) Collect sequences and discover all step task strings
#     ds_sequences: List[Tuple[str, List[Dict]]] = []
#     all_tasks = set()

#     txt_files = sorted(ACTION_DIR.glob("P*.txt"))
#     print(f"Found {len(txt_files)} action files")

#     for txt_path in tqdm(txt_files, desc="Building multiview sequences"):
#         instance_id = txt_path.stem
#         try:
#             mv_seq = build_multiview_sequence_for_instance(instance_id,
#                                                            fps=fps,
#                                                            max_frames=max_frames,
#                                                            conf=conf)
#         except Exception as e:
#             print(f"Skipping {instance_id} due to error: {e}")
#             continue
#         if not mv_seq:
#             continue
#         for step in mv_seq:
#             a = step["action"]
#             if a and a.task:
#                 all_tasks.add(a.task)
#         ds_sequences.append((instance_id, mv_seq))

#     print(f"Collected {len(ds_sequences)} sequences")
#     print(f"Discovered {len(all_tasks)} distinct step tasks")
#     task_to_id = {task: i for i, task in enumerate(sorted(all_tasks))}

#     # 2) Convert each sequence to (X_i, y_i)
#     X_list: List[np.ndarray] = []
#     y_list: List[np.ndarray] = []
#     seq_instance_ids: List[str] = []

#     for instance_id, mv_seq in ds_sequences:
#         T = len(mv_seq)
#         X = np.zeros((T, feature_dim), dtype=np.float32)
#         y = np.full((T,), -1, dtype=np.int64)
#         for t, step in enumerate(mv_seq):
#             X[t] = counts_to_vector(step["counts_fused"])
#             a = step["action"]
#             if a and a.task and a.task in task_to_id:
#                 y[t] = task_to_id[a.task]
#         if (y != -1).any():  # keep only sequences with at least one labeled frame
#             X_list.append(X)
#             y_list.append(y)
#             seq_instance_ids.append(instance_id)

#     print(f"Prepared {len(X_list)} sequences with at least one labeled frame")
#     return X_list, y_list, seq_instance_ids, task_to_id


# # Build the dataset (you can adjust fps/max_frames later if needed)
# X_list, y_list, seq_instance_ids, task_to_id = build_multiview_dataset(
#     fps=1.0,
#     max_frames=120,
#     conf=0.25,
# )

# print("Example sequence shapes:", [x.shape for x in X_list[:3]])
# print("Number of unique step labels:", len(task_to_id))

In [188]:
# PyTorch dataset + training loop for step recognition on GPU

from torch.utils.data import Dataset, DataLoader


class StepSequenceDataset(Dataset):
    def __init__(self, X_list: List[np.ndarray], y_list: List[np.ndarray]):
        assert len(X_list) == len(y_list)
        self.X_list = X_list
        self.y_list = y_list

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        return self.X_list[idx], self.y_list[idx]


def collate_sequences(batch):
    """Pad variable-length sequences in a batch.

    Returns:
        X_pad: (B, T_max, D)
        y_pad: (B, T_max) with -1 for unlabeled / padded frames
        mask: (B, T_max) bool mask for labeled frames
    """
    lengths = [x.shape[0] for x, _ in batch]
    B = len(batch)
    T_max = max(lengths)

    X_pad = torch.zeros(B, T_max, feature_dim, dtype=torch.float32)
    y_pad = torch.full((B, T_max), -1, dtype=torch.long)
    mask = torch.zeros(B, T_max, dtype=torch.bool)

    for i, (x, y) in enumerate(batch):
        L = x.shape[0]
        X_pad[i, :L] = torch.from_numpy(x)
        y_torch = torch.from_numpy(y)
        y_pad[i, :L] = y_torch
        mask[i, :L] = (y_torch != -1)

    return X_pad, y_pad, mask


# Simple train/val split by sequence
num_sequences = len(X_list)
indices = np.arange(num_sequences)
np.random.seed(0)
np.random.shuffle(indices)

split = int(0.8 * num_sequences)
train_idx = indices[:split]
val_idx = indices[split:]

X_train = [X_list[i] for i in train_idx]
Y_train = [y_list[i] for i in train_idx]
X_val = [X_list[i] for i in val_idx]
Y_val = [y_list[i] for i in val_idx]

train_ds = StepSequenceDataset(X_train, Y_train)
val_ds = StepSequenceDataset(X_val, Y_val)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_sequences)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_sequences)


# Instantiate model for step recognition
num_step_classes = len(task_to_id)
input_dim = feature_dim
hidden_dim = 128

step_model = SimpleGRUHead(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_step_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
step_model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = optim.Adam(step_model.parameters(), lr=1e-3)


def evaluate_step_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for Xb, yb, mask in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mask = mask.to(device)
            logits = model(Xb)  # (B, T, C)
            preds = logits.argmax(dim=-1)
            # Only consider frames with labels (mask True)
            correct += ((preds == yb) & mask).sum().item()
            total += mask.sum().item()
    acc = correct / total if total > 0 else 0.0
    return acc


# Training loop
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    step_model.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yb, mask in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        mask = mask.to(device)

        logits = step_model(Xb)  # (B, T, C)
        B, T, C = logits.shape
        loss = criterion(logits.view(B * T, C), yb.view(B * T))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    train_loss = total_loss / max(n_batches, 1)
    val_acc = evaluate_step_model(step_model, val_loader)
    print(f"Epoch {epoch}/{num_epochs} - train_loss={train_loss:.4f}, val_step_acc={val_acc:.3f}")

print("Done training step recognition model.")

Epoch 1/10 - train_loss=2.8489, val_step_acc=0.201
Epoch 2/10 - train_loss=2.4005, val_step_acc=0.232
Epoch 3/10 - train_loss=2.2013, val_step_acc=0.347
Epoch 4/10 - train_loss=1.9276, val_step_acc=0.339
Epoch 5/10 - train_loss=1.7909, val_step_acc=0.331
Epoch 6/10 - train_loss=1.6599, val_step_acc=0.395
Epoch 7/10 - train_loss=1.5596, val_step_acc=0.453
Epoch 8/10 - train_loss=1.4584, val_step_acc=0.404
Epoch 9/10 - train_loss=1.3668, val_step_acc=0.450
Epoch 10/10 - train_loss=1.2741, val_step_acc=0.460
Done training step recognition model.


## Diagnose low validation accuracy

47.9% accuracy on 23 step classes is low. Let's check:
1. Class distribution (imbalance?)
2. Label coverage (how many frames are actually labeled?)
3. Feature quality (are YOLO counts discriminative enough?)
4. Model predictions vs ground truth

In [189]:
# Diagnose the low accuracy issue

# 1. Check class distribution
from collections import Counter

all_labels = []
for y in y_list:
    labeled = y[y != -1]
    all_labels.extend(labeled.tolist())

label_counts = Counter(all_labels)
print("Step class distribution:")
for label_id, count in sorted(label_counts.items()):
    task_name = [k for k, v in task_to_id.items() if v == label_id][0]
    print(f"  {label_id:2d} ({task_name:30s}): {count:5d} frames ({100*count/len(all_labels):.1f}%)")

print(f"\nTotal labeled frames: {len(all_labels)}")
print(f"Most common class: {label_counts.most_common(1)[0][1]} frames ({100*label_counts.most_common(1)[0][1]/len(all_labels):.1f}%)")
print(f"Random baseline (23 classes): {100/23:.1f}%")

# 2. Check label coverage per sequence
labeled_ratios = []
for y in y_list:
    ratio = (y != -1).sum() / len(y)
    labeled_ratios.append(ratio)

print(f"\nLabel coverage per sequence:")
print(f"  Mean: {np.mean(labeled_ratios):.2%}")
print(f"  Min: {np.min(labeled_ratios):.2%}")
print(f"  Max: {np.max(labeled_ratios):.2%}")

# 3. Check feature statistics
all_features = np.vstack(X_list)
print(f"\nFeature statistics:")
print(f"  Mean non-zero features per frame: {(all_features > 0).sum(axis=1).mean():.1f}")
print(f"  Max count per class: {all_features.max():.0f}")
print(f"  Mean count per class: {all_features[all_features > 0].mean():.2f}")

# 4. Check validation predictions
step_model.eval()
val_preds = []
val_labels = []
with torch.inference_mode():
    for Xb, yb, mask in val_loader:
        Xb = Xb.to(device)
        logits = step_model(Xb)
        preds = logits.argmax(dim=-1).cpu()
        for i in range(Xb.shape[0]):
            valid = mask[i].cpu()
            val_preds.extend(preds[i][valid].numpy())
            val_labels.extend(yb[i][valid].cpu().numpy())

from sklearn.metrics import confusion_matrix, classification_report

print("\nValidation confusion matrix (top 10 classes by frequency):")
top_classes = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:10]
top_class_ids = [c[0] for c in top_classes]
val_preds_arr = np.array(val_preds)
val_labels_arr = np.array(val_labels)

# Filter to top classes only
mask_top = np.isin(val_labels_arr, top_class_ids)
if mask_top.sum() > 0:
    cm = confusion_matrix(val_labels_arr[mask_top], val_preds_arr[mask_top], labels=top_class_ids)
    print("Rows=ground truth, Cols=predicted")
    for i, label_id in enumerate(top_class_ids):
        task_name = [k for k, v in task_to_id.items() if v == label_id][0]
        print(f"  {label_id:2d} {task_name[:25]:25s}: {cm[i].sum():4d} true, pred={top_class_ids[cm[i].argmax()]:2d} (acc={100*cm[i,i]/max(cm[i].sum(),1):.1f}%)")

Step class distribution:
   0 (add_70pct_ethanol             ):   870 frames (3.3%)
   1 (add_binding_buffer            ):   913 frames (3.5%)
   2 (add_cell_lystate              ):  1557 frames (5.9%)
   3 (add_magnetic_beads            ):   469 frames (1.8%)
   4 (add_pbs                       ):  2567 frames (9.7%)
   5 (add_wash_buffer               ):  1039 frames (3.9%)
   6 (aspirate_pbs                  ):  2136 frames (8.1%)
   7 (aspirate_supernatant          ):  1601 frames (6.1%)
   8 (dispense_solution             ):   293 frames (1.1%)
   9 (pipetting                     ):  1574 frames (6.0%)
  10 (place_in_magnetic_rack        ):   407 frames (1.5%)
  11 (remove_culture_medium         ):  2130 frames (8.1%)
  12 (shake_plate                   ):  1150 frames (4.4%)
  13 (spindown                      ):  1636 frames (6.2%)
  14 (transfer_cell_lystate_to_tube ):   847 frames (3.2%)
  15 (transfer_forward_primer_to_8_tube_stripes):   739 frames (2.8%)
  16 (transfer_pcrmi

In [190]:
# Improvements: Better features + class weighting + longer training

# 1. Normalize features (L2 norm per frame) to reduce scale issues
def normalize_features(X_list):
    X_norm = []
    for X in X_list:
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1.0  # avoid division by zero
        X_norm.append(X / norms)
    return X_norm

# 2. Compute class weights for imbalanced classes
from sklearn.utils.class_weight import compute_class_weight

all_train_labels = []
for y in Y_train:
    labeled = y[y != -1]
    all_train_labels.extend(labeled.tolist())

if len(all_train_labels) > 0:
    unique_labels = np.unique(all_train_labels)
    class_weights = compute_class_weight('balanced', classes=unique_labels, y=all_train_labels)
    weight_dict = {int(unique_labels[i]): float(class_weights[i]) for i in range(len(unique_labels))}
    # Create weight tensor for CrossEntropyLoss
    weights_tensor = torch.ones(num_step_classes, dtype=torch.float32)
    for label_id, weight in weight_dict.items():
        weights_tensor[label_id] = weight
    weights_tensor = weights_tensor.to(device)
    print("Computed class weights for imbalanced classes")
else:
    weights_tensor = None

# 3. Use normalized features
X_train_norm = normalize_features(X_train)
X_val_norm = normalize_features(X_val)

train_ds_norm = StepSequenceDataset(X_train_norm, Y_train)
val_ds_norm = StepSequenceDataset(X_val_norm, Y_val)

train_loader_norm = DataLoader(train_ds_norm, batch_size=4, shuffle=True, collate_fn=collate_sequences)
val_loader_norm = DataLoader(val_ds_norm, batch_size=4, shuffle=False, collate_fn=collate_sequences)

# 4. Larger model + better training
step_model_v2 = SimpleGRUHead(input_dim=input_dim, hidden_dim=256, num_classes=num_step_classes, num_layers=2)
step_model_v2.to(device)

criterion_v2 = nn.CrossEntropyLoss(ignore_index=-1, weight=weights_tensor)
optimizer_v2 = optim.Adam(step_model_v2.parameters(), lr=5e-4)

print("Training improved model (normalized features, class weights, larger hidden=256, 2 layers)...")

num_epochs_v2 = 20
best_val_acc = 0.0

for epoch in range(1, num_epochs_v2 + 1):
    step_model_v2.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yb, mask in train_loader_norm:
        Xb = Xb.to(device)
        yb = yb.to(device)

        logits = step_model_v2(Xb)
        B, T, C = logits.shape
        loss = criterion_v2(logits.view(B * T, C), yb.view(B * T))

        optimizer_v2.zero_grad()
        loss.backward()
        optimizer_v2.step()

        total_loss += loss.item()
        n_batches += 1

    train_loss = total_loss / max(n_batches, 1)
    val_acc = evaluate_step_model(step_model_v2, val_loader_norm)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
    print(f"Epoch {epoch}/{num_epochs_v2} - train_loss={train_loss:.4f}, val_step_acc={val_acc:.3f} (best={best_val_acc:.3f})")

print(f"\nBest validation accuracy: {best_val_acc:.3f}")

Computed class weights for imbalanced classes
Training improved model (normalized features, class weights, larger hidden=256, 2 layers)...


Epoch 1/20 - train_loss=3.1248, val_step_acc=0.107 (best=0.107)
Epoch 2/20 - train_loss=2.8870, val_step_acc=0.210 (best=0.210)
Epoch 3/20 - train_loss=2.4789, val_step_acc=0.263 (best=0.263)
Epoch 4/20 - train_loss=2.2722, val_step_acc=0.298 (best=0.298)
Epoch 5/20 - train_loss=2.5220, val_step_acc=0.175 (best=0.298)
Epoch 6/20 - train_loss=2.1522, val_step_acc=0.319 (best=0.319)
Epoch 7/20 - train_loss=2.0221, val_step_acc=0.248 (best=0.319)
Epoch 8/20 - train_loss=2.0229, val_step_acc=0.196 (best=0.319)
Epoch 9/20 - train_loss=1.9156, val_step_acc=0.261 (best=0.319)
Epoch 10/20 - train_loss=1.8169, val_step_acc=0.262 (best=0.319)
Epoch 11/20 - train_loss=1.9212, val_step_acc=0.233 (best=0.319)
Epoch 12/20 - train_loss=1.9457, val_step_acc=0.261 (best=0.319)
Epoch 13/20 - train_loss=1.7955, val_step_acc=0.323 (best=0.323)
Epoch 14/20 - train_loss=1.7240, val_step_acc=0.323 (best=0.323)
Epoch 15/20 - train_loss=1.6966, val_step_acc=0.255 (best=0.323)
Epoch 16/20 - train_loss=1.7178, v

## Root cause: Raw YOLO counts are too weak

The problem is that **just counting objects per frame** doesn't capture:
1. **Which objects are being manipulated** (hand-object proximity)
2. **Temporal dynamics** (what changes between frames)
3. **Spatial relationships** (objects near each other)

Let's build **much richer features** that include temporal deltas and hand-object interactions.

In [191]:
# Build MUCH richer features: temporal deltas + hand-object interactions
# (Fast version - works directly on existing X_list without reloading videos)
print("Building rich features from existing sequences (temporal deltas + hand-object heuristics)...")

def enrich_features_fast(X_list, y_list):
    """Build rich features from existing X arrays.
    
    Features per frame:
    - Original: 35-dim YOLO counts
    - Temporal delta: 35-dim change from previous frame
    - Hand presence: 2-dim (left_hand, right_hand counts)
    - Object diversity: 1-dim (number of unique object types)
    - Total: 35 + 35 + 2 + 1 = 73 dims
    """
    X_list_rich = []
    left_hand_idx = name_to_index.get('left_hand', -1)
    right_hand_idx = name_to_index.get('right_hand', -1)
    
    for X in X_list:
        T, D = X.shape
        rich_dim = D + D + 2 + 1  # counts + deltas + hands + diversity
        X_rich = np.zeros((T, rich_dim), dtype=np.float32)
        
        # Base counts
        X_rich[:, :D] = X
        
        # Temporal deltas
        X_rich[1:, D:D*2] = X[1:] - X[:-1]
        
        # Hand presence (normalized)
        if left_hand_idx >= 0:
            X_rich[:, D*2] = X[:, left_hand_idx] / (1.0 + X[:, left_hand_idx].max())
        if right_hand_idx >= 0:
            X_rich[:, D*2+1] = X[:, right_hand_idx] / (1.0 + X[:, right_hand_idx].max())
        
        # Object diversity (how many different object types detected)
        X_rich[:, D*2+2] = (X > 0).sum(axis=1) / D
        
        X_list_rich.append(X_rich)
    
    return X_list_rich

X_list_rich = enrich_features_fast(X_list, y_list)
feature_dim_rich = X_list_rich[0].shape[1]
print(f"Rich feature dimension: {feature_dim_rich} (was {feature_dim})")

# Update feature_dim
feature_dim_rich = X_list_rich[0].shape[1]
print(f"Rich feature dimension: {feature_dim_rich} (was {feature_dim})")

# Re-split with rich features
X_train_rich = [X_list_rich[i] for i in train_idx]
X_val_rich = [X_list_rich[i] for i in val_idx]

train_ds_rich = StepSequenceDataset(X_train_rich, Y_train)
val_ds_rich = StepSequenceDataset(X_val_rich, Y_val)

train_loader_rich = DataLoader(train_ds_rich, batch_size=4, shuffle=True, collate_fn=collate_sequences)
val_loader_rich = DataLoader(val_ds_rich, batch_size=4, shuffle=False, collate_fn=collate_sequences)

# Update collate function to use new feature_dim
def collate_sequences_rich(batch):
    lengths = [x.shape[0] for x, _ in batch]
    B = len(batch)
    T_max = max(lengths)
    
    X_pad = torch.zeros(B, T_max, feature_dim_rich, dtype=torch.float32)
    y_pad = torch.full((B, T_max), -1, dtype=torch.long)
    mask = torch.zeros(B, T_max, dtype=torch.bool)
    
    for i, (x, y) in enumerate(batch):
        L = x.shape[0]
        X_pad[i, :L] = torch.from_numpy(x)
        y_torch = torch.from_numpy(y)
        y_pad[i, :L] = y_torch
        mask[i, :L] = (y_torch != -1)
    
    return X_pad, y_pad, mask

train_loader_rich = DataLoader(train_ds_rich, batch_size=4, shuffle=True, collate_fn=collate_sequences_rich)
val_loader_rich = DataLoader(val_ds_rich, batch_size=4, shuffle=False, collate_fn=collate_sequences_rich)

Building rich features from existing sequences (temporal deltas + hand-object heuristics)...
Rich feature dimension: 73 (was 35)
Rich feature dimension: 73 (was 35)


In [192]:
# Add manipulated object detection: which objects are near hands (using bbox proximity)

def compute_manipulated_objects(dets_fpv, dets_tpv, counts_fpv, counts_tpv):
    """Detect which objects are being manipulated based on hand-object bbox proximity.
    
    Returns:
        manipulated_left: (35,) binary vector - objects near left hand
        manipulated_right: (35,) binary vector - objects near right hand
        manipulation_strength: (35,) float - IoU/overlap scores
    """
    manipulated_left = np.zeros(feature_dim, dtype=np.float32)
    manipulated_right = np.zeros(feature_dim, dtype=np.float32)
    manipulation_strength = np.zeros(feature_dim, dtype=np.float32)
    
    def iou(bbox1, bbox2):
        x1_min, y1_min, x1_max, y1_max = bbox1
        x2_min, y2_min, x2_max, y2_max = bbox2
        inter_x_min = max(x1_min, x2_min)
        inter_y_min = max(y1_min, y2_min)
        inter_x_max = min(x1_max, x2_max)
        inter_y_max = min(y1_max, y2_max)
        if inter_x_max <= inter_x_min or inter_y_max <= inter_y_min:
            return 0.0
        inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
        area1 = (x1_max - x1_min) * (y1_max - y1_min)
        area2 = (x2_max - x2_min) * (y2_max - y2_min)
        union_area = area1 + area2 - inter_area
        return inter_area / union_area if union_area > 0 else 0.0
    
    def distance(bbox1, bbox2):
        x1_min, y1_min, x1_max, y1_max = bbox1
        x2_min, y2_min, x2_max, y2_max = bbox2
        c1_x, c1_y = (x1_min + x1_max) / 2, (y1_min + y1_max) / 2
        c2_x, c2_y = (x2_min + x2_max) / 2, (y2_min + y2_max) / 2
        return np.sqrt((c1_x - c2_x)**2 + (c1_y - c2_y)**2)
    
    # Collect hand bboxes from both views
    left_hand_bboxes = []
    right_hand_bboxes = []
    all_dets = dets_fpv + dets_tpv
    
    for det in all_dets:
        if det['class_name'] == 'left_hand':
            left_hand_bboxes.append(det['bbox'])
        elif det['class_name'] == 'right_hand':
            right_hand_bboxes.append(det['bbox'])
    
    # For each object detection, check proximity to hands
    for det in all_dets:
        obj_name = det['class_name']
        if obj_name in ['left_hand', 'right_hand']:
            continue
        obj_idx = name_to_index.get(obj_name)
        if obj_idx is None:
            continue
        
        obj_bbox = det['bbox']
        max_iou_left = 0.0
        max_iou_right = 0.0
        min_dist_left = float('inf')
        min_dist_right = float('inf')
        
        for hand_bbox in left_hand_bboxes:
            iou_val = iou(obj_bbox, hand_bbox)
            dist = distance(obj_bbox, hand_bbox)
            max_iou_left = max(max_iou_left, iou_val)
            min_dist_left = min(min_dist_left, dist)
        
        for hand_bbox in right_hand_bboxes:
            iou_val = iou(obj_bbox, hand_bbox)
            dist = distance(obj_bbox, hand_bbox)
            max_iou_right = max(max_iou_right, iou_val)
            min_dist_right = min(min_dist_right, dist)
        
        # If IoU > 0.1 or distance < threshold, mark as manipulated
        # Normalize distance by image size (assuming ~640x480, normalize to [0,1])
        dist_threshold = 100.0  # pixels
        if max_iou_left > 0.1 or min_dist_left < dist_threshold:
            manipulated_left[obj_idx] = 1.0
            manipulation_strength[obj_idx] = max(manipulation_strength[obj_idx], max_iou_left)
        if max_iou_right > 0.1 or min_dist_right < dist_threshold:
            manipulated_right[obj_idx] = 1.0
            manipulation_strength[obj_idx] = max(manipulation_strength[obj_idx], max_iou_right)
    
    return manipulated_left, manipulated_right, manipulation_strength


# Rebuild sequences with manipulated object features (need full mv_seq dicts)
print("Rebuilding sequences with manipulated object detection...")

def build_sequences_with_manipulation():
    """Rebuild multiview sequences and add manipulated object features."""
    X_list_manip = []
    txt_files = sorted(ACTION_DIR.glob("P*.txt"))
    
    for txt_path in tqdm(txt_files[:len(X_list)], desc="Adding manipulation features"):
        instance_id = txt_path.stem
        try:
            mv_seq = build_multiview_sequence_for_instance(instance_id, fps=1.0, max_frames=120, conf=0.25)
            if not mv_seq:
                continue
            
            T = len(mv_seq)
            # Base features: counts + deltas + hands + diversity (73 dims)
            # Build counts array first
            counts_array = np.array([counts_to_vector(step["counts_fused"]) for step in mv_seq])
            X_base_list = enrich_features_fast([counts_array], [])
            X_base = X_base_list[0]  # Get the array
            
            # Add manipulation features (35 + 35 + 35 = 105 dims)
            manip_dim = feature_dim * 3  # left_manip, right_manip, strength
            X_manip = np.zeros((T, manip_dim), dtype=np.float32)
            
            for t, step in enumerate(mv_seq):
                left_manip, right_manip, strength = compute_manipulated_objects(
                    step["dets_fpv"], step["dets_tpv"],
                    step["counts_fpv"], step["counts_tpv"]
                )
                X_manip[t, :feature_dim] = left_manip
                X_manip[t, feature_dim:feature_dim*2] = right_manip
                X_manip[t, feature_dim*2:] = strength
            
            # Concatenate: base (73) + manipulation (105) = 178 dims
            X_full = np.concatenate([X_base, X_manip], axis=1)
            X_list_manip.append(X_full)
        except Exception as e:
            print(f"Skipping {instance_id}: {e}")
            # Fallback: use original features padded to expected manip dim (73 base + 105 manip = 178)
            orig_idx = seq_instance_ids.index(instance_id) if instance_id in seq_instance_ids else 0
            orig_X = X_list[orig_idx]
            T_orig = orig_X.shape[0]
            expected_manip_dim = 73 + 105  # base + manipulation
            X_fallback = np.zeros((T_orig, expected_manip_dim), dtype=np.float32)
            # Use enriched features if available, else pad original
            if orig_idx < len(X_list_rich):
                X_fallback[:, :73] = X_list_rich[orig_idx]
            else:
                X_fallback[:, :feature_dim] = orig_X
            X_list_manip.append(X_fallback)
    
    return X_list_manip

X_list_with_manip = build_sequences_with_manipulation()
feature_dim_manip = X_list_with_manip[0].shape[1]
print(f"Feature dimension with manipulation: {feature_dim_manip}")

# Re-split
X_train_manip = [X_list_with_manip[i] for i in train_idx]
X_val_manip = [X_list_with_manip[i] for i in val_idx]

train_ds_manip = StepSequenceDataset(X_train_manip, Y_train)
val_ds_manip = StepSequenceDataset(X_val_manip, Y_val)

def collate_sequences_manip(batch):
    lengths = [x.shape[0] for x, _ in batch]
    B = len(batch)
    T_max = max(lengths)
    X_pad = torch.zeros(B, T_max, feature_dim_manip, dtype=torch.float32)
    y_pad = torch.full((B, T_max), -1, dtype=torch.long)
    mask = torch.zeros(B, T_max, dtype=torch.bool)
    for i, (x, y) in enumerate(batch):
        L = x.shape[0]
        X_pad[i, :L] = torch.from_numpy(x)
        y_torch = torch.from_numpy(y)
        y_pad[i, :L] = y_torch
        mask[i, :L] = (y_torch != -1)
    return X_pad, y_pad, mask

train_loader_manip = DataLoader(train_ds_manip, batch_size=4, shuffle=True, collate_fn=collate_sequences_manip)
val_loader_manip = DataLoader(val_ds_manip, batch_size=4, shuffle=False, collate_fn=collate_sequences_manip)

Rebuilding sequences with manipulated object detection...


Adding manipulation features:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded 129 actions for P01_01_01
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_01.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_01_T1.mp4
FPV: fps=29.97, frames=4339, dur=144.78s
TPV: fps=29.97, frames=4339, dur=144.78s


Adding manipulation features:   0%|          | 1/226 [00:28<1:48:19, 28.89s/it]

Built multiview sequence of length 120 for P01_01_01
Loaded 145 actions for P01_01_02
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_02.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_02_T1.mp4
FPV: fps=29.97, frames=3767, dur=125.69s
TPV: fps=29.97, frames=3767, dur=125.69s


...

Built multiview sequence of length 120 for P32_05_01
Loaded 285 actions for P32_06_01
Using FPV: /home/nyan/FineBio/11 finebio_videos_fpv_valid/finebio_videos/P32_06_01.mp4
Using TPV: /home/nyan/FineBio/06 finebio_videos_tpv_valid/finebio_videos/P32_06_01_T1.mp4
FPV: fps=29.97, frames=9417, dur=314.21s
TPV: fps=29.97, frames=9417, dur=314.21s


Adding manipulation features: 100%|█████████▉| 225/226 [1:46:01<00:28, 28.70s/it]

Built multiview sequence of length 120 for P32_06_01
Loaded 307 actions for P32_07_01
Using FPV: /home/nyan/FineBio/11 finebio_videos_fpv_valid/finebio_videos/P32_07_01.mp4
Using TPV: /home/nyan/FineBio/06 finebio_videos_tpv_valid/finebio_videos/P32_07_01_T1.mp4
FPV: fps=29.97, frames=9471, dur=316.02s
TPV: fps=29.97, frames=9471, dur=316.02s


Adding manipulation features: 100%|██████████| 226/226 [1:46:30<00:00, 28.28s/it]

Built multiview sequence of length 120 for P32_07_01
Feature dimension with manipulation: 178


In [193]:
# Train with manipulation features

step_model_manip = SimpleGRUHead(input_dim=feature_dim_manip, hidden_dim=256, num_classes=num_step_classes, num_layers=2)
step_model_manip.to(device)

criterion_manip = nn.CrossEntropyLoss(ignore_index=-1, weight=weights_tensor)
optimizer_manip = optim.Adam(step_model_manip.parameters(), lr=3e-4)

def evaluate_step_model_manip(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for Xb, yb, mask in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mask = mask.to(device)
            logits = model(Xb)
            preds = logits.argmax(dim=-1)
            correct += ((preds == yb) & mask).sum().item()
            total += mask.sum().item()
    return correct / total if total > 0 else 0.0

print("Training with manipulation features...")
print(f"Input dim: {feature_dim_manip}, Hidden: 256, Classes: {num_step_classes}")

num_epochs_manip = 25
best_val_acc_manip = 0.0

for epoch in range(1, num_epochs_manip + 1):
    step_model_manip.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yb, mask in train_loader_manip:
        Xb = Xb.to(device)
        yb = yb.to(device)
        
        logits = step_model_manip(Xb)
        B, T, C = logits.shape
        loss = criterion_manip(logits.view(B * T, C), yb.view(B * T))
        
        optimizer_manip.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(step_model_manip.parameters(), max_norm=1.0)
        optimizer_manip.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    train_loss = total_loss / max(n_batches, 1)
    val_acc = evaluate_step_model_manip(step_model_manip, val_loader_manip)
    if val_acc > best_val_acc_manip:
        best_val_acc_manip = val_acc
    print(f"Epoch {epoch}/{num_epochs_manip} - train_loss={train_loss:.4f}, val_step_acc={val_acc:.3f} (best={best_val_acc_manip:.3f})")

print(f"\n=== RESULTS WITH MANIPULATION DETECTION ===")
print(f"Best validation accuracy: {best_val_acc_manip:.3f} ({best_val_acc_manip*100:.1f}%)")
print(f"Previous best (rich features): 0.638 (63.8%)")
print(f"Improvement: {best_val_acc_manip - 0.638:.3f} ({100*(best_val_acc_manip - 0.638)/0.638:.1f}% relative)")

Training with manipulation features...
Input dim: 178, Hidden: 256, Classes: 23
Epoch 1/25 - train_loss=3.0256, val_step_acc=0.205 (best=0.205)
Epoch 2/25 - train_loss=2.5928, val_step_acc=0.288 (best=0.288)
Epoch 3/25 - train_loss=2.2323, val_step_acc=0.300 (best=0.300)
Epoch 4/25 - train_loss=1.9515, val_step_acc=0.356 (best=0.356)
Epoch 5/25 - train_loss=1.7963, val_step_acc=0.381 (best=0.381)
Epoch 6/25 - train_loss=1.5803, val_step_acc=0.405 (best=0.405)
Epoch 7/25 - train_loss=1.3922, val_step_acc=0.455 (best=0.455)
Epoch 8/25 - train_loss=1.2010, val_step_acc=0.532 (best=0.532)
Epoch 9/25 - train_loss=1.0207, val_step_acc=0.500 (best=0.532)
Epoch 10/25 - train_loss=0.9580, val_step_acc=0.578 (best=0.578)
Epoch 11/25 - train_loss=0.8009, val_step_acc=0.568 (best=0.578)
Epoch 12/25 - train_loss=0.6664, val_step_acc=0.605 (best=0.605)
Epoch 13/25 - train_loss=0.5876, val_step_acc=0.619 (best=0.619)
Epoch 14/25 - train_loss=0.5150, val_step_acc=0.586 (best=0.619)
Epoch 15/25 - train

## Export YOLO-based features + atomic annotations for ActionFormer

This will write one `.npy` feature file per FineBio instance and one JSON
with temporal annotations (verb / manipulated / affected / atomic).

In [198]:
from pathlib import Path
import json

EXPORT_ROOT = Path("/home/nyan/k-project/finebio_actionformer")
FEATURE_DIR = EXPORT_ROOT / "features_yolo26"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_PATH = EXPORT_ROOT / "finebio_atomic_yolo.json"

print("Export root:", EXPORT_ROOT)


def infer_subset_from_fpv(instance_id: str) -> str:
    fpv_paths = resolve_video_paths(instance_id, view="fpv")
    for p in fpv_paths:
        s = str(p)
        if "09 finebio_videos_fpv_train" in s:
            return "training"
        if "11 finebio_videos_fpv_valid" in s:
            return "validation"
        if "04 finebio_videos_fpv_test" in s:
            return "test"
    return "training"


def get_video_duration_sec(instance_id: str) -> float:
    fpv_paths = resolve_video_paths(instance_id, view="fpv")
    if not fpv_paths:
        return 0.0
    p = fpv_paths[0]
    cap = cv2.VideoCapture(str(p))
    if not cap.isOpened():
        return 0.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    return n / fps if fps > 0 else 0.0


def export_for_actionformer(fps: float = 1.0):
    db = {}
    txt_files = sorted(ACTION_DIR.glob("P*.txt"))
    print("Exporting", len(txt_files), "instances")

    for txt_path in txt_files:
        instance_id = txt_path.stem
        print("Instance:", instance_id)

        # 1) Build multiview sequence with YOLO detections
        mv_seq = build_multiview_sequence_for_instance(instance_id, fps=fps, max_frames=None, conf=0.25)
        if not mv_seq:
            continue

        # 2) Build feature matrix (T, D) using existing counts_to_vector + enrich_features_fast
        counts_array = np.array([counts_to_vector(step["counts_fused"]) for step in mv_seq])
        X_rich_list = enrich_features_fast([counts_array], [])
        X_feat = X_rich_list[0]  # (T, feature_dim_rich)

        # 3) Save features
        feat_path = FEATURE_DIR / f"{instance_id}.npy"
        np.save(feat_path, X_feat)

        # 4) Build annotation entry
        actions = load_actions_for_instance(instance_id)
        duration = get_video_duration_sec(instance_id)
        subset = infer_subset_from_fpv(instance_id)

        annos = []
        for a in actions:
            atomic_label = "{}__{}__{}".format(a.verb or "", a.manipulated_object or "", a.affected_object or "")
            annos.append({
                "segment": [float(a.start_sec), float(a.end_sec)],
                "verb": a.verb,
                "manipulated_object": a.manipulated_object,
                "affected_object": a.affected_object,
                "atomic": atomic_label,
            })

        db[instance_id] = {
            "subset": subset,
            "duration": float(duration),
            "feature_path": str(feat_path),
            "annotations": annos,
        }

    out = {"database": db}
    ANNOT_PATH.write_text(json.dumps(out, indent=2))
    print("Wrote annotations to", ANNOT_PATH)


# Run this once (will take a while: YOLO over all videos)
export_for_actionformer(fps=1.0)

Export root: /home/nyan/k-project/finebio_actionformer
Exporting 226 instances
Instance: P01_01_01
Loaded 129 actions for P01_01_01
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_01.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_01_T1.mp4
FPV: fps=29.97, frames=4339, dur=144.78s
TPV: fps=29.97, frames=4339, dur=144.78s
Built multiview sequence of length 145 for P01_01_01
Loaded 129 actions for P01_01_01
Instance: P01_01_02
Loaded 145 actions for P01_01_02
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_02.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_02_T1.mp4
FPV: fps=29.97, frames=3767, dur=125.69s
TPV: fps=29.97, frames=3767, dur=125.69s
Built multiview sequence of length 126 for P01_01_02
Loaded 145 actions for P01_01_02
Instance: P01_02_01
Loaded 214 actions for P01_02_01
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos

## Multi-task GRU: verb / manipulated / affected / atomic (frame-level)

We reuse the rich YOLO features and train **one GRU** with four heads:
verb, manipulated object, affected object, and atomic operation.

In [201]:
# Build vocabularies and frame-level labels for verb / manipulated / affected / atomic

fps_dataset = 1.0  # we sampled multiview sequences at 1 fps

# 1) Collect vocabularies
verb_set = set()
manip_set = set()
aff_set = set()
atomic_set = set()

for instance_id in seq_instance_ids:
    actions = load_actions_for_instance(instance_id)
    for a in actions:
        if a.verb:
            verb_set.add(a.verb)
        if a.manipulated_object:
            manip_set.add(a.manipulated_object)
        if a.affected_object:
            aff_set.add(a.affected_object)
        atomic_label = f"{a.verb or ''}__{a.manipulated_object or ''}__{a.affected_object or ''}"
        atomic_set.add(atomic_label)

verb_list = sorted(verb_set)
manip_list = sorted(manip_set)
aff_list = sorted(aff_set)
atomic_list = sorted(atomic_set)

verb_to_id = {v: i for i, v in enumerate(verb_list)}
manip_to_id = {v: i for i, v in enumerate(manip_list)}
aff_to_id = {v: i for i, v in enumerate(aff_list)}
atomic_to_id = {v: i for i, v in enumerate(atomic_list)}

print("#verbs", len(verb_list), "#manip", len(manip_list), "#aff", len(aff_list), "#atomic", len(atomic_list))

# 2) Build per-frame label sequences aligned with X_list_rich

# Use rich features if available; else fall back to X_list
feature_source = X_list_rich if 'X_list_rich' in globals() else X_list
feature_dim_mt = feature_source[0].shape[1]
print("Multi-task feature dim:", feature_dim_mt)

Y_verb_list = []
Y_manip_list = []
Y_aff_list = []
Y_atomic_list = []

for X, instance_id in zip(feature_source, seq_instance_ids):
    T = X.shape[0]
    yv = np.full(T, -1, dtype=np.int64)
    ym = np.full(T, -1, dtype=np.int64)
    ya = np.full(T, -1, dtype=np.int64)
    yat = np.full(T, -1, dtype=np.int64)

    actions = load_actions_for_instance(instance_id)
    # For faster lookup, just linear scan (datasets are small enough)
    for t in range(T):
        time_sec = t / fps_dataset
        chosen = None
        for a in actions:
            if a.start_sec <= time_sec <= a.end_sec:
                chosen = a
                break
        if not chosen:
            continue
        if chosen.verb in verb_to_id:
            yv[t] = verb_to_id[chosen.verb]
        if chosen.manipulated_object in manip_to_id:
            ym[t] = manip_to_id[chosen.manipulated_object]
        if chosen.affected_object in aff_to_id:
            ya[t] = aff_to_id[chosen.affected_object]
        atomic_label = f"{chosen.verb or ''}__{chosen.manipulated_object or ''}__{chosen.affected_object or ''}"
        if atomic_label in atomic_to_id:
            yat[t] = atomic_to_id[atomic_label]

    Y_verb_list.append(yv)
    Y_manip_list.append(ym)
    Y_aff_list.append(ya)
    Y_atomic_list.append(yat)

print("Built multi-task labels for", len(feature_source), "sequences")

Loaded 129 actions for P01_01_01
Loaded 145 actions for P01_01_02
Loaded 214 actions for P01_02_01
Loaded 210 actions for P01_02_02
Loaded 290 actions for P01_03_01
Loaded 249 actions for P01_03_02
Loaded 306 actions for P01_04_01
Loaded 317 actions for P01_04_02
Loaded 343 actions for P01_05_01
Loaded 363 actions for P01_05_02
Loaded 156 actions for P02_01_01
Loaded 150 actions for P02_01_02
Loaded 199 actions for P02_02_01
Loaded 196 actions for P02_02_02
Loaded 288 actions for P02_03_01
Loaded 289 actions for P02_03_02
Loaded 347 actions for P02_04_01
Loaded 336 actions for P02_04_02
Loaded 307 actions for P02_05_01
Loaded 300 actions for P02_05_02
Loaded 146 actions for P03_01_01
Loaded 144 actions for P03_01_02
Loaded 169 actions for P03_02_01
Loaded 188 actions for P03_02_02
Loaded 237 actions for P03_03_01
Loaded 228 actions for P03_03_02
Loaded 275 actions for P03_04_01
Loaded 292 actions for P03_04_02
Loaded 308 actions for P03_05_01
Loaded 290 actions for P03_05_02
Loaded 178

In [202]:
# Multi-head GRU model and training

from torch.utils.data import Dataset, DataLoader

class MultiTaskSequenceDataset(Dataset):
    def __init__(self, X_list, Yv_list, Ym_list, Ya_list, Yat_list):
        self.X_list = X_list
        self.Yv_list = Yv_list
        self.Ym_list = Ym_list
        self.Ya_list = Ya_list
        self.Yat_list = Yat_list

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        return (self.X_list[idx],
                self.Yv_list[idx],
                self.Ym_list[idx],
                self.Ya_list[idx],
                self.Yat_list[idx])


def collate_multitask(batch):
    lengths = [x.shape[0] for x, _, _, _, _ in batch]
    B = len(batch)
    T_max = max(lengths)

    X_pad = torch.zeros(B, T_max, feature_dim_mt, dtype=torch.float32)
    yv_pad = torch.full((B, T_max), -1, dtype=torch.long)
    ym_pad = torch.full((B, T_max), -1, dtype=torch.long)
    ya_pad = torch.full((B, T_max), -1, dtype=torch.long)
    yat_pad = torch.full((B, T_max), -1, dtype=torch.long)

    for i, (x, yv, ym, ya, yat) in enumerate(batch):
        L = x.shape[0]
        X_pad[i, :L] = torch.from_numpy(x)
        yv_pad[i, :L] = torch.from_numpy(yv)
        ym_pad[i, :L] = torch.from_numpy(ym)
        ya_pad[i, :L] = torch.from_numpy(ya)
        yat_pad[i, :L] = torch.from_numpy(yat)

    return X_pad, yv_pad, ym_pad, ya_pad, yat_pad


class MultiHeadGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim,
                 n_verbs, n_manip, n_aff, n_atomic,
                 num_layers: int = 1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, bidirectional=True)
        hd2 = hidden_dim * 2
        self.head_verb = nn.Linear(hd2, n_verbs)
        self.head_manip = nn.Linear(hd2, n_manip)
        self.head_aff = nn.Linear(hd2, n_aff)
        self.head_atomic = nn.Linear(hd2, n_atomic)

    def forward(self, x):
        h, _ = self.gru(x)  # (B, T, 2H)
        return {
            'verb': self.head_verb(h),
            'manip': self.head_manip(h),
            'aff': self.head_aff(h),
            'atomic': self.head_atomic(h),
        }


# Train/val split
num_sequences_mt = len(feature_source)
indices = np.arange(num_sequences_mt)
np.random.seed(0)
np.random.shuffle(indices)
split = int(0.8 * num_sequences_mt)
train_idx_mt = indices[:split]
val_idx_mt = indices[split:]

X_train_mt = [feature_source[i] for i in train_idx_mt]
Yv_train = [Y_verb_list[i] for i in train_idx_mt]
Ym_train = [Y_manip_list[i] for i in train_idx_mt]
Ya_train = [Y_aff_list[i] for i in train_idx_mt]
Yat_train = [Y_atomic_list[i] for i in train_idx_mt]

X_val_mt = [feature_source[i] for i in val_idx_mt]
Yv_val = [Y_verb_list[i] for i in val_idx_mt]
Ym_val = [Y_manip_list[i] for i in val_idx_mt]
Ya_val = [Y_aff_list[i] for i in val_idx_mt]
Yat_val = [Y_atomic_list[i] for i in val_idx_mt]

train_ds_mt = MultiTaskSequenceDataset(X_train_mt, Yv_train, Ym_train, Ya_train, Yat_train)
val_ds_mt = MultiTaskSequenceDataset(X_val_mt, Yv_val, Ym_val, Ya_val, Yat_val)

train_loader_mt = DataLoader(train_ds_mt, batch_size=4, shuffle=True, collate_fn=collate_multitask)
val_loader_mt = DataLoader(val_ds_mt, batch_size=4, shuffle=False, collate_fn=collate_multitask)

# Instantiate model
n_verbs = len(verb_list)
n_manip = len(manip_list)
n_aff = len(aff_list)
n_atomic = len(atomic_list)

mt_model = MultiHeadGRU(input_dim=feature_dim_mt, hidden_dim=256,
                        n_verbs=n_verbs, n_manip=n_manip,
                        n_aff=n_aff, n_atomic=n_atomic, num_layers=1).to(device)

crit = nn.CrossEntropyLoss(ignore_index=-1)
opt_mt = optim.Adam(mt_model.parameters(), lr=1e-3)


def eval_multitask(model, loader):
    model.eval()
    correct_v = correct_m = correct_a = correct_at = 0
    total_v = total_m = total_a = total_at = 0
    with torch.inference_mode():
        for Xb, yv, ym, ya, yat in loader:
            Xb = Xb.to(device)
            yv = yv.to(device)
            ym = ym.to(device)
            ya = ya.to(device)
            yat = yat.to(device)
            out = model(Xb)
            for head, y, corr, tot in [
                ('verb', yv, 'v', 'tv'),
            ]:
                pass
    # quick and dirty: we'll just compute atomic accuracy here for now
    atomic_correct = 0
    atomic_total = 0
    model.eval()
    with torch.inference_mode():
        for Xb, _, _, _, yat in loader:
            Xb = Xb.to(device)
            yat = yat.to(device)
            out = model(Xb)['atomic']
            preds = out.argmax(dim=-1)
            mask = (yat != -1)
            atomic_correct += ((preds == yat) & mask).sum().item()
            atomic_total += mask.sum().item()
    acc_atomic = atomic_correct / atomic_total if atomic_total > 0 else 0.0
    return acc_atomic


num_epochs_mt = 10
for epoch in range(1, num_epochs_mt + 1):
    mt_model.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yv, ym, ya, yat in train_loader_mt:
        Xb = Xb.to(device)
        yv = yv.to(device)
        ym = ym.to(device)
        ya = ya.to(device)
        yat = yat.to(device)

        out = mt_model(Xb)
        loss = 0.0
        for logits, y in [(out['verb'], yv), (out['manip'], ym), (out['aff'], ya), (out['atomic'], yat)]:
            B, T, C = logits.shape
            loss = loss + crit(logits.view(B * T, C), y.view(B * T))

        opt_mt.zero_grad()
        loss.backward()
        opt_mt.step()

        total_loss += loss.item()
        n_batches += 1

    train_loss = total_loss / max(n_batches, 1)
    val_atomic_acc = eval_multitask(mt_model, val_loader_mt)
    print(f"Epoch {epoch}/{num_epochs_mt} - train_loss={train_loss:.4f}, val_atomic_acc={val_atomic_acc:.3f}")

Epoch 1/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 2/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 3/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 4/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 5/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 6/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 7/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 8/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 9/10 - train_loss=nan, val_atomic_acc=1.000
Epoch 10/10 - train_loss=nan, val_atomic_acc=1.000


In [206]:
# Frame-level mAP for manipulated / affected object presence

from collections import defaultdict
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

print("Computing frame-level manipulated / affected mAP (this will re-run YOLO over videos)...")

manip_scores = defaultdict(list)  # class_name -> list of scores
manip_labels = defaultdict(list)
aff_scores = defaultdict(list)
aff_labels = defaultdict(list)

# classes we care about (from annotations)
manip_classes = manip_list
aff_classes = aff_list

for instance_id in tqdm(seq_instance_ids, desc="Instances"):
    actions = load_actions_for_instance(instance_id)
    if not actions:
        continue
    # multiview sequence (has counts_fpv/tpv + dets_fpv/tpv)
    mv_seq = build_multiview_sequence_for_instance(instance_id, fps=1.0, max_frames=120, conf=0.25)
    if not mv_seq:
        continue

    for step in mv_seq:
        t = step["time_sec"]
        # find active action at this time
        a = assign_action_label(t, actions)
        manip_gt = a.manipulated_object if a else None
        aff_gt = a.affected_object if a else None

        # manipulated: use existing IoU-based strength
        left_manip, right_manip, strength = compute_manipulated_objects(
            step["dets_fpv"], step["dets_tpv"],
            step["counts_fpv"], step["counts_tpv"],
        )
        for name in manip_classes:
            idx = name_to_index.get(name)
            if idx is None:
                continue
            score = float(strength[idx])  # 0..1
            label = 1 if manip_gt == name else 0
            manip_scores[name].append(score)
            manip_labels[name].append(label)

        # affected: score = summed detection confidence per class
        conf_per_class = defaultdict(float)
        for det in (step["dets_fpv"] + step["dets_tpv"]):
            conf_per_class[det["class_name"]] += float(det["score"])
        for name in aff_classes:
            score = float(conf_per_class.get(name, 0.0))
            label = 1 if aff_gt == name else 0
            aff_scores[name].append(score)
            aff_labels[name].append(label)

# compute per-class AP and mAP
manip_aps = []
aff_aps = []

for name in manip_classes:
    y_true = np.array(manip_labels[name])
    y_score = np.array(manip_scores[name])
    if y_true.sum() == 0 or len(np.unique(y_true)) < 2:
        continue
    try:
        ap = average_precision_score(y_true, y_score)
    except Exception:
        ap = 0.0
    manip_aps.append(ap)

for name in aff_classes:
    y_true = np.array(aff_labels[name])
    y_score = np.array(aff_scores[name])
    if y_true.sum() == 0 or len(np.unique(y_true)) < 2:
        continue
    try:
        ap = average_precision_score(y_true, y_score)
    except Exception:
        ap = 0.0
    aff_aps.append(ap)

mAP_manip = float(np.mean(manip_aps)) if manip_aps else 0.0
mAP_aff = float(np.mean(aff_aps)) if aff_aps else 0.0

print(f"Manipulated object frame-level mAP: {mAP_manip*100:.2f}% over {len(manip_aps)} classes")
print(f"Affected object frame-level mAP: {mAP_aff*100:.2f}% over {len(aff_aps)} classes")

Computing frame-level manipulated / affected mAP (this will re-run YOLO over videos)...


Instances:   0%|          | 0/226 [00:00<?, ?it/s]

Loaded 129 actions for P01_01_01
Loaded 129 actions for P01_01_01
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_01.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_01_T1.mp4
FPV: fps=29.97, frames=4339, dur=144.78s
TPV: fps=29.97, frames=4339, dur=144.78s


Instances:   0%|          | 1/226 [00:29<1:49:53, 29.31s/it]

Built multiview sequence of length 120 for P01_01_01
Loaded 145 actions for P01_01_02
Loaded 145 actions for P01_01_02
Using FPV: /home/nyan/FineBio/09 finebio_videos_fpv_train/finebio_videos/P01_01_02.mp4
Using TPV: /home/nyan/FineBio/05 finebio_videos_tpv_train/finebio_videos/P01_01_02_T1.mp4
FPV: fps=29.97, frames=3767, dur=125.69s
TPV: fps=29.97, frames=3767, dur=125.69s


...

Built multiview sequence of length 120 for P32_05_01
Loaded 285 actions for P32_06_01
Loaded 285 actions for P32_06_01
Using FPV: /home/nyan/FineBio/11 finebio_videos_fpv_valid/finebio_videos/P32_06_01.mp4
Using TPV: /home/nyan/FineBio/06 finebio_videos_tpv_valid/finebio_videos/P32_06_01_T1.mp4
FPV: fps=29.97, frames=9417, dur=314.21s
TPV: fps=29.97, frames=9417, dur=314.21s


Instances: 100%|█████████▉| 225/226 [1:46:25<00:28, 28.49s/it]

Built multiview sequence of length 120 for P32_06_01
Loaded 307 actions for P32_07_01
Loaded 307 actions for P32_07_01
Using FPV: /home/nyan/FineBio/11 finebio_videos_fpv_valid/finebio_videos/P32_07_01.mp4
Using TPV: /home/nyan/FineBio/06 finebio_videos_tpv_valid/finebio_videos/P32_07_01_T1.mp4
FPV: fps=29.97, frames=9471, dur=316.02s
TPV: fps=29.97, frames=9471, dur=316.02s


Instances: 100%|██████████| 226/226 [1:46:54<00:00, 28.38s/it]

Built multiview sequence of length 120 for P32_07_01
Manipulated object frame-level mAP: 0.01% over 1 classes
Affected object frame-level mAP: 0.00% over 0 classes


In [208]:
# Multi-head GRU (step + verb + manipulated + affected) with per-head frame-Accuracy + frame-mAP

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score

# --- Inputs (assumed already created earlier in this notebook) ---
# feature_source: List[np.ndarray]  -> each (T, D)
# seq_instance_ids: List[str]       -> same length
# y_list: List[np.ndarray]          -> step labels per frame (T,) with -1
# task_to_id: Dict[str,int]

assert len(feature_source) == len(seq_instance_ids)
assert len(y_list) == len(feature_source)

D_in = feature_source[0].shape[1]
num_step = len(task_to_id)
print('Using feature dim:', D_in)
print('Num step classes:', num_step)

fps_dataset = 1.0

# Build verb/manip/aff vocab with min frequency to avoid ultra-rare labels killing learning
MIN_FRAMES_PER_CLASS = 100

verb_counts = {}
manip_counts = {}
aff_counts = {}

def _inc(d, k):
    if k is None or k == "":
        return
    d[k] = d.get(k, 0) + 1

# First pass: count labeled frames per class
for X, instance_id in zip(feature_source, seq_instance_ids):
    T = X.shape[0]
    actions = load_actions_for_instance(instance_id)
    for t in range(T):
        time_sec = t / fps_dataset
        a = assign_action_label(time_sec, actions)
        if not a:
            continue
        _inc(verb_counts, a.verb)
        _inc(manip_counts, a.manipulated_object)
        _inc(aff_counts, a.affected_object)

verb_list = sorted([k for k, c in verb_counts.items() if c >= MIN_FRAMES_PER_CLASS])
manip_list = sorted([k for k, c in manip_counts.items() if c >= MIN_FRAMES_PER_CLASS])
aff_list = sorted([k for k, c in aff_counts.items() if c >= MIN_FRAMES_PER_CLASS])

verb_to_id = {v: i for i, v in enumerate(verb_list)}
manip_to_id = {v: i for i, v in enumerate(manip_list)}
aff_to_id = {v: i for i, v in enumerate(aff_list)}

print('Verb classes kept:', len(verb_list))
print('Manip classes kept:', len(manip_list))
print('Aff classes kept:', len(aff_list))

# Second pass: build per-frame labels (aligned with feature_source)
Y_step = y_list
Y_verb = []
Y_manip = []
Y_aff = []

for X, instance_id in zip(feature_source, seq_instance_ids):
    T = X.shape[0]
    yv = np.full(T, -1, dtype=np.int64)
    ym = np.full(T, -1, dtype=np.int64)
    ya = np.full(T, -1, dtype=np.int64)

    actions = load_actions_for_instance(instance_id)
    for t in range(T):
        time_sec = t / fps_dataset
        a = assign_action_label(time_sec, actions)
        if not a:
            continue
        if a.verb in verb_to_id:
            yv[t] = verb_to_id[a.verb]
        if a.manipulated_object in manip_to_id:
            ym[t] = manip_to_id[a.manipulated_object]
        if a.affected_object in aff_to_id:
            ya[t] = aff_to_id[a.affected_object]

    Y_verb.append(yv)
    Y_manip.append(ym)
    Y_aff.append(ya)


class MTSeqDS(Dataset):
    def __init__(self, X, ys, yv, ym, ya):
        self.X = X
        self.ys = ys
        self.yv = yv
        self.ym = ym
        self.ya = ya
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.ys[i], self.yv[i], self.ym[i], self.ya[i]


def collate_mt(batch):
    lens = [b[0].shape[0] for b in batch]
    B = len(batch)
    Tm = max(lens)
    Xp = torch.zeros(B, Tm, D_in, dtype=torch.float32)
    ys = torch.full((B, Tm), -1, dtype=torch.long)
    yv = torch.full((B, Tm), -1, dtype=torch.long)
    ym = torch.full((B, Tm), -1, dtype=torch.long)
    ya = torch.full((B, Tm), -1, dtype=torch.long)
    for i, (x, _ys, _yv, _ym, _ya) in enumerate(batch):
        L = x.shape[0]
        Xp[i, :L] = torch.from_numpy(x)
        ys[i, :L] = torch.from_numpy(_ys)
        yv[i, :L] = torch.from_numpy(_yv)
        ym[i, :L] = torch.from_numpy(_ym)
        ya[i, :L] = torch.from_numpy(_ya)
    return Xp, ys, yv, ym, ya


class MultiHeadGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_step, n_verb, n_manip, n_aff):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        hd2 = hidden_dim * 2
        self.h_step = nn.Linear(hd2, n_step)
        self.h_verb = nn.Linear(hd2, n_verb)
        self.h_manip = nn.Linear(hd2, n_manip)
        self.h_aff = nn.Linear(hd2, n_aff)

    def forward(self, x):
        h, _ = self.gru(x)
        return {
            'step': self.h_step(h),
            'verb': self.h_verb(h),
            'manip': self.h_manip(h),
            'aff': self.h_aff(h),
        }


def class_weights_from_seq(Y_seq_list, num_classes):
    counts = np.zeros(num_classes, dtype=np.int64)
    for y in Y_seq_list:
        yy = y[y != -1]
        for c in yy:
            counts[int(c)] += 1
    w = np.ones(num_classes, dtype=np.float32)
    nz = counts > 0
    w[nz] = counts[nz].sum() / (counts[nz] * nz.sum())
    return torch.tensor(w, dtype=torch.float32)


def head_metrics(logits, y_true):
    # logits: (B,T,C), y_true: (B,T)
    with torch.inference_mode():
        C = logits.shape[-1]
        y = y_true.view(-1)
        l = logits.view(-1, C)
        mask = (y != -1)
        if mask.sum().item() == 0:
            return 0.0, 0.0, 0
        y = y[mask]
        l = l[mask]
        preds = l.argmax(dim=-1)
        acc = (preds == y).float().mean().item()

        probs = torch.softmax(l, dim=-1).cpu().numpy()
        y_np = y.cpu().numpy()
        aps = []
        for c in range(C):
            yt = (y_np == c).astype(np.int32)
            if yt.sum() == 0 or yt.sum() == len(yt):
                continue
            aps.append(average_precision_score(yt, probs[:, c]))
        mAP = float(np.mean(aps)) if aps else 0.0
        return acc, mAP, len(aps)


# Split
N = len(feature_source)
idx = np.arange(N)
np.random.seed(0)
np.random.shuffle(idx)
sp = int(0.8 * N)
tr, va = idx[:sp], idx[sp:]

X_tr = [feature_source[i] for i in tr]
X_va = [feature_source[i] for i in va]
Ys_tr = [Y_step[i] for i in tr]
Ys_va = [Y_step[i] for i in va]
Yv_tr = [Y_verb[i] for i in tr]
Yv_va = [Y_verb[i] for i in va]
Ym_tr = [Y_manip[i] for i in tr]
Ym_va = [Y_manip[i] for i in va]
Ya_tr = [Y_aff[i] for i in tr]
Ya_va = [Y_aff[i] for i in va]

train_loader = DataLoader(MTSeqDS(X_tr, Ys_tr, Yv_tr, Ym_tr, Ya_tr), batch_size=4, shuffle=True, collate_fn=collate_mt)
val_loader = DataLoader(MTSeqDS(X_va, Ys_va, Yv_va, Ym_va, Ya_va), batch_size=4, shuffle=False, collate_fn=collate_mt)

# Model
n_verb = max(1, len(verb_list))
n_manip = max(1, len(manip_list))
n_aff = max(1, len(aff_list))

model = MultiHeadGRU(D_in, 256, num_step, n_verb, n_manip, n_aff).to(device)

w_step = class_weights_from_seq(Ys_tr, num_step).to(device)
w_verb = class_weights_from_seq(Yv_tr, n_verb).to(device)
w_manip = class_weights_from_seq(Ym_tr, n_manip).to(device)
w_aff = class_weights_from_seq(Ya_tr, n_aff).to(device)

ce_step = nn.CrossEntropyLoss(ignore_index=-1, weight=w_step)
ce_verb = nn.CrossEntropyLoss(ignore_index=-1, weight=w_verb)
ce_manip = nn.CrossEntropyLoss(ignore_index=-1, weight=w_manip)
ce_aff = nn.CrossEntropyLoss(ignore_index=-1, weight=w_aff)

opt = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for ep in range(1, EPOCHS + 1):
    model.train()
    tot = 0.0
    nb = 0
    for Xb, ys, yv, ym, ya in train_loader:
        Xb = Xb.to(device)
        ys = ys.to(device)
        yv = yv.to(device)
        ym = ym.to(device)
        ya = ya.to(device)
        out = model(Xb)
        loss = (
            ce_step(out['step'].reshape(-1, num_step), ys.view(-1))
            + ce_verb(out['verb'].reshape(-1, n_verb), yv.view(-1))
            + ce_manip(out['manip'].reshape(-1, n_manip), ym.view(-1))
            + ce_aff(out['aff'].reshape(-1, n_aff), ya.view(-1))
        )
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += loss.item()
        nb += 1

    model.eval()
    step_acc = step_map = verb_acc = verb_map = manip_acc = manip_map = aff_acc = aff_map = 0.0
    step_k = verb_k = manip_k = aff_k = 0
    with torch.inference_mode():
        for Xb, ys, yv, ym, ya in val_loader:
            Xb = Xb.to(device)
            ys = ys.to(device)
            yv = yv.to(device)
            ym = ym.to(device)
            ya = ya.to(device)
            out = model(Xb)

            a, m, k = head_metrics(out['step'], ys); step_acc += a; step_map += m; step_k += k
            a, m, k = head_metrics(out['verb'], yv); verb_acc += a; verb_map += m; verb_k += k
            a, m, k = head_metrics(out['manip'], ym); manip_acc += a; manip_map += m; manip_k += k
            a, m, k = head_metrics(out['aff'], ya); aff_acc += a; aff_map += m; aff_k += k

    denom = len(val_loader)
    print(
        f"ep {ep}/{EPOCHS} loss={tot/max(nb,1):.3f} | "
        f"step acc={step_acc/denom:.3f} mAP={step_map/denom:.3f} | "
        f"verb acc={verb_acc/denom:.3f} mAP={verb_map/denom:.3f} | "
        f"manip acc={manip_acc/denom:.3f} mAP={manip_map/denom:.3f} | "
        f"aff acc={aff_acc/denom:.3f} mAP={aff_map/denom:.3f}"
    )


Using feature dim: 73
Num step classes: 23
Loaded 129 actions for P01_01_01
Loaded 145 actions for P01_01_02
Loaded 214 actions for P01_02_01
Loaded 210 actions for P01_02_02
Loaded 290 actions for P01_03_01
Loaded 249 actions for P01_03_02
Loaded 306 actions for P01_04_01
Loaded 317 actions for P01_04_02
Loaded 343 actions for P01_05_01
Loaded 363 actions for P01_05_02
Loaded 156 actions for P02_01_01
Loaded 150 actions for P02_01_02
Loaded 199 actions for P02_02_01
Loaded 196 actions for P02_02_02
Loaded 288 actions for P02_03_01
Loaded 289 actions for P02_03_02
Loaded 347 actions for P02_04_01
Loaded 336 actions for P02_04_02
Loaded 307 actions for P02_05_01
Loaded 300 actions for P02_05_02
Loaded 146 actions for P03_01_01
Loaded 144 actions for P03_01_02
Loaded 169 actions for P03_02_01
Loaded 188 actions for P03_02_02
Loaded 237 actions for P03_03_01
Loaded 228 actions for P03_03_02
Loaded 275 actions for P03_04_01
Loaded 292 actions for P03_04_02
Loaded 308 actions for P03_05_01


In [203]:
import numpy as np, glob
import_path = "/home/nyan/k-project/finebio_actionformer/features_yolo26/*.npy"
sample = np.load(glob.glob(import_path)[0])
print(sample.shape)  # (T, D) -> use D as FEAT_DIM

(111, 73)


In [207]:
# Train with rich features + better architecture

step_model_rich = SimpleGRUHead(input_dim=feature_dim_rich, hidden_dim=256, num_classes=num_step_classes, num_layers=2)
step_model_rich.to(device)

criterion_rich = nn.CrossEntropyLoss(ignore_index=-1, weight=weights_tensor)
optimizer_rich = optim.Adam(step_model_rich.parameters(), lr=3e-4)

def evaluate_step_model_rich(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for Xb, yb, mask in loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mask = mask.to(device)
            logits = model(Xb)
            preds = logits.argmax(dim=-1)
            correct += ((preds == yb) & mask).sum().item()
            total += mask.sum().item()
    return correct / total if total > 0 else 0.0

print("Training with rich features (temporal deltas + hand-object interactions)...")
print(f"Input dim: {feature_dim_rich}, Hidden: 256, Classes: {num_step_classes}")

num_epochs_rich = 25
best_val_acc_rich = 0.0

for epoch in range(1, num_epochs_rich + 1):
    step_model_rich.train()
    total_loss = 0.0
    n_batches = 0
    for Xb, yb, mask in train_loader_rich:
        Xb = Xb.to(device)
        yb = yb.to(device)
        
        logits = step_model_rich(Xb)
        B, T, C = logits.shape
        loss = criterion_rich(logits.view(B * T, C), yb.view(B * T))
        
        optimizer_rich.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(step_model_rich.parameters(), max_norm=1.0)  # gradient clipping
        optimizer_rich.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    train_loss = total_loss / max(n_batches, 1)
    val_acc = evaluate_step_model_rich(step_model_rich, val_loader_rich)
    if val_acc > best_val_acc_rich:
        best_val_acc_rich = val_acc
    print(f"Epoch {epoch}/{num_epochs_rich} - train_loss={train_loss:.4f}, val_step_acc={val_acc:.3f} (best={best_val_acc_rich:.3f})")

print(f"\n=== FINAL RESULTS ===")
print(f"Best validation accuracy with rich features: {best_val_acc_rich:.3f} ({best_val_acc_rich*100:.1f}%)")
print(f"Previous best (raw counts): 0.479 (47.9%)")
print(f"Improvement: {best_val_acc_rich - 0.479:.3f} ({100*(best_val_acc_rich - 0.479)/0.479:.1f}% relative)")

Training with rich features (temporal deltas + hand-object interactions)...
Input dim: 73, Hidden: 256, Classes: 23
Epoch 1/25 - train_loss=3.0545, val_step_acc=0.131 (best=0.131)
Epoch 2/25 - train_loss=2.6094, val_step_acc=0.257 (best=0.257)
Epoch 3/25 - train_loss=2.2493, val_step_acc=0.268 (best=0.268)
Epoch 4/25 - train_loss=2.0103, val_step_acc=0.381 (best=0.381)
Epoch 5/25 - train_loss=1.8479, val_step_acc=0.429 (best=0.429)
Epoch 6/25 - train_loss=1.7250, val_step_acc=0.315 (best=0.429)
Epoch 7/25 - train_loss=1.5624, val_step_acc=0.375 (best=0.429)
Epoch 8/25 - train_loss=1.4288, val_step_acc=0.404 (best=0.429)
Epoch 9/25 - train_loss=1.2706, val_step_acc=0.509 (best=0.509)
Epoch 10/25 - train_loss=1.1498, val_step_acc=0.484 (best=0.509)
Epoch 11/25 - train_loss=1.0938, val_step_acc=0.547 (best=0.547)
Epoch 12/25 - train_loss=0.9223, val_step_acc=0.552 (best=0.552)
Epoch 13/25 - train_loss=0.8527, val_step_acc=0.571 (best=0.571)
Epoch 14/25 - train_loss=0.7530, val_step_acc=0.

In [205]:
model = ultralytics.YOLO("/home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt")
metrics = model.val(data="/home/nyan/k-project/finebio_yolo_joint/finebio_joint_test.yaml", split="test")
print(metrics.box.map50, metrics.box.map)  # mAP50, mAP50-95

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15843MiB)
YOLO26n summary (fused): 122 layers, 2,381,661 parameters, 0 gradients, 5.2 GFLOPs


FileNotFoundError: '/home/nyan/k-project/finebio_yolo_joint/finebio_joint_test.yaml' does not exist